# 🌳 10 · Decision trees, random forests and cancer driver mutations
### Entropy, information gain and ID3 from scratch; overfitting and pruning; continuous splits; the variance problem and how a forest of randomised trees solves it — ending with CHASM, a random forest that separates driver from passenger mutations

**BiologyToML** · Level 1 · 02-620 core · Part B — Supervised learning · Module 10 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/10_decision_trees_random_forests.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib; scikit-learn (a logistic-regression baseline and for checking our own trees and forests) |
| **Prerequisites** | Modules 01–02 (NumPy, probability, logarithms) and 08 (classification, training vs test). Module 09 (logistic regression) helps for Section 8. |
| **Estimated time** | 165 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Decision trees and random forests: trees as non-probabilistic classifiers (internal nodes, branches, leaves; the PlayTennis example), greedy top-down learning (ID3, C4.5), sample entropy, conditional entropy and information gain = mutual information, noisy data and overfitting, early stopping and reduced-error pruning, continuous inputs and thresholds, trees as axis-parallel rectangles and IF-THEN rules, pros and cons (high variance), random forests (bootstrap + random feature subsets + majority vote), and the CHASM case study of driver vs passenger mutations |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> PlayTennis (Mitchell 1997) is the classic 14-day teaching table, typed into the course toolkit. Every other data set is
> **SIMULATED** by code: the flow-cytometry blood cells of Sections 7–8 (`simulate_blood_cells`), the breast-tumour
> nuclear features of Section 9 (`simulate_tumor_features`, shaped like the Wisconsin diagnostic data), the
> genotype–disease study of Sections 5–6, the two-marker cells of Section 7.6 and the "mutation" data of Section 10.
> They show mechanisms and do not reproduce any study. The CHASM numbers quoted in Section 10 are the published ones
> (Carter *et al.* 2009).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a decision tree is (internal nodes test attributes, branches are values, leaves predict) and why biologists like trees (interactions, interpretability, cheap tests).
2. **Compute** entropy, conditional entropy and information gain by hand and in code, reproducing every number of the PlayTennis worked example.
3. **Implement** the ID3 algorithm recursively from scratch and **translate** a tree into IF-THEN rules.
4. **Diagnose** overfitting (one noisy example, label noise in a simulated genotype study) and **implement** reduced-error pruning with a validation set.
5. **Handle** continuous inputs with thresholds and **interpret** trees as axis-parallel rectangles; **contrast** them with a linear boundary.
6. **Demonstrate** that trees are high-variance models and **derive** why averaging decorrelated trees helps, $\rho\sigma^2+(1-\rho)\sigma^2/B$.
7. **Build** a random forest (bootstrap + random feature subsets + majority vote) with an out-of-bag error estimate, and **connect** it to CHASM's driver-mutation classifier.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, course toolkit, simulated data | 5 min |
| 1 · What a decision tree is | the PlayTennis tree as a function and as a diagram; why trees | 10 min |
| 2 · Entropy | impurity, bits, the coding interpretation | 15 min |
| 3 · Information gain | conditional entropy, mutual information, choosing the root | 20 min |
| 4 · ID3 from scratch | recursion, the learned tree, IF-THEN rules | 15 min |
| 5 · Noise and overfitting | D15; a simulated epistasis study; accuracy vs tree size | 15 min |
| 6 · Pruning | early stopping vs reduced-error pruning | 15 min |
| 7 · Continuous inputs | thresholds, rectangles on flow-cytometry data (automated gating), where trees struggle | 15 min |
| 8 · High variance | one extra cell; bootstrap resamples; tree vs logistic regression | 10 min |
| 9 · Random forests | variance of an average, our own forest, OOB error, m and depth | 25 min |
| 10 · Case study: CHASM | driver vs passenger mutations; a simulated mini-CHASM | 15 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

A colorectal tumour can carry tens of somatic mutations in protein-coding genes; a melanoma or lung tumour from a
smoker, hundreds. Only a handful of them are **drivers** — changes that give the cell a growth advantage and are
positively selected, such as a missense mutation that switches off the tumour suppressor *TP53*. The rest are
**passengers**: they happened to be present in the cell that expanded, and they ride along without functional
consequence. Telling the two apart matters for understanding how cancers arise and for choosing targeted therapies.

How would a biologist decide by hand? Probably with a series of yes/no questions: *Is the residue conserved across
species? Is it in a functional domain? Has this exact change been seen in other tumours? Does it swap a hydrophobic
for a charged amino acid?* That is a **decision tree**. In 2009 Carter and colleagues built **CHASM**, which combines
hundreds of such trees — a **random forest** — trained on 2,488 known driver missense mutations versus synthetic
passengers; on held-out data it called 195 of 196 experimentally observed *TP53* mutations drivers (Section 10).

This notebook builds that machinery from the ground up: first the entropy-based tree learner of the lecture (ID3) on
Tom Mitchell's classic PlayTennis table, then the problems that come with it (overfitting, high variance), and finally
the random forest that fixes them. Decision trees are one of the course's **non-probabilistic** methods: there is no
likelihood to maximise — a tree is grown greedily by an information criterion. (Its probabilistic cousins are the
classifiers of Modules 08–09; a leaf can still report class *frequencies*, P(Y | X ∈ leaf).)

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: apart from the 14-day PlayTennis table
> (a textbook example, typed in), every data set in this notebook is generated by code. Where the 02-620 lecture
> used Fisher's iris flowers to draw tree boundaries, we use `simulate_blood_cells` — 150 white blood cells measured
> by flow cytometry (forward scatter, side scatter, CD45, CD14), three cell types, one easy to separate and two that
> overlap a little, just like the three iris species. Where many courses use the Wisconsin breast-cancer biopsies, we
> use `simulate_tumor_features`, 569 simulated tumours with the same 30 nuclear measurements (label 1 = malignant). A
> simulator also has a known truth — in the genotype study of Section 5 we know the disease rule we planted, so we can
> check whether a pruned tree recovers it. None of the numbers below are the original studies' numbers.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import copy
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.tree import DecisionTreeClassifier

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`load_play_tennis` returns the classic 14-day PlayTennis table (and the noisy day D15), `simulate_genotypes` draws 0/1/2 genotypes for the simulated study of Section 5, `simulate_blood_cells` and `simulate_tumor_features` generate the SIMULATED flow-cytometry and breast-tumour data of Sections 7–9, and the remaining helpers score classifiers, split data and draw decision regions. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PLAY_TENNIS_ROWS = [
    ("D1", "Sunny", "Hot", "High", "Weak", "No"),
    ("D2", "Sunny", "Hot", "High", "Strong", "No"),
    ("D3", "Overcast", "Hot", "High", "Weak", "Yes"),
    ("D4", "Rain", "Mild", "High", "Weak", "Yes"),
    ("D5", "Rain", "Cool", "Normal", "Weak", "Yes"),
    ("D6", "Rain", "Cool", "Normal", "Strong", "No"),
    ("D7", "Overcast", "Cool", "Normal", "Strong", "Yes"),
    ("D8", "Sunny", "Mild", "High", "Weak", "No"),
    ("D9", "Sunny", "Cool", "Normal", "Weak", "Yes"),
    ("D10", "Rain", "Mild", "Normal", "Weak", "Yes"),
    ("D11", "Sunny", "Mild", "Normal", "Strong", "Yes"),
    ("D12", "Overcast", "Mild", "High", "Strong", "Yes"),
    ("D13", "Overcast", "Hot", "Normal", "Weak", "Yes"),
    ("D14", "Rain", "Mild", "High", "Strong", "No"),
]


PLAY_TENNIS_D15 = ("D15", "Sunny", "Hot", "Normal", "Strong", "No")


def load_play_tennis(noisy: bool = False) -> pd.DataFrame:
    """PlayTennis training data (14 days); ``noisy=True`` appends the mislabelled day D15."""
    rows = PLAY_TENNIS_ROWS + ([PLAY_TENNIS_D15] if noisy else [])
    return pd.DataFrame(rows, columns=["Day", "Outlook", "Temperature", "Humidity", "Wind", "PlayTennis"])


def simulate_tumor_features(seed: int = 1995) -> dict:
    """Synthetic breast-tumour nuclear features shaped like the Wisconsin diagnostic data set.

    569 fine-needle aspirates (212 malignant, 357 benign).  For 10 properties of the cell nuclei in
    each image — radius, texture, perimeter, area, smoothness, compactness, concavity, concave
    points, symmetry, fractal dimension — three summaries are given: the mean, the standard error
    ("error") and the largest value ("worst"), i.e. 30 features.  Geometry ties them together
    (perimeter ≈ 2π·radius, area ≈ π·radius²) and a shared shape-irregularity factor links
    compactness, concavity and concave points, so many features are strongly correlated.

    Returns a dict with ``X`` (DataFrame, 569 × 30), ``y`` (1 = malignant, 0 = benign),
    ``target_names`` and ``source``.
    """
    rng = np.random.default_rng(seed)
    y = rng.permutation(np.r_[np.ones(212, int), np.zeros(357, int)])
    n = len(y)
    m = y == 1
    radius = np.where(m, rng.normal(17.3, 3.0, n), rng.normal(12.2, 1.8, n)).clip(6.5, 28)
    texture = np.where(m, rng.normal(21.5, 3.8, n), rng.normal(17.9, 4.0, n)).clip(9, 40)
    irregular = rng.normal(0, 1, n) + np.where(m, 2.3, 0.0)               # shared shape factor
    smooth = (0.092 + 0.008 * m + 0.012 * rng.normal(0, 1, n) + 0.004 * irregular).clip(0.05, 0.17)
    compact = np.exp(-2.5 + 0.38 * irregular + 0.2 * rng.normal(0, 1, n))
    concav = np.exp(-3.4 + 0.75 * irregular + 0.35 * rng.normal(0, 1, n))
    cpoints = np.exp(-3.8 + 0.6 * irregular + 0.25 * rng.normal(0, 1, n))
    symmetry = (0.175 + 0.008 * irregular + 0.025 * rng.normal(0, 1, n)).clip(0.1, 0.3)
    fractal = (0.063 + 0.002 * irregular - 0.0004 * (radius - 14) + 0.006 * rng.normal(0, 1, n)).clip(0.045, 0.1)
    perimeter = 2 * np.pi * radius * (1 + 0.08 * compact / compact.mean()) + rng.normal(0, 1.5, n)
    area = np.pi * radius ** 2 * (0.97 + 0.03 * rng.normal(0, 1, n))
    means = dict(radius=radius, texture=texture, perimeter=perimeter, area=area, smoothness=smooth,
                 compactness=compact, concavity=concav, **{"concave points": cpoints}, symmetry=symmetry,
                 **{"fractal dimension": fractal})
    cols = {}
    spread = np.exp(rng.normal(0, 0.35, n)) * (1 + 0.6 * m)              # heterogeneity of the nuclei
    for name, v in means.items():
        cols[f"mean {name}"] = v
    for name, v in means.items():
        cols[f"{name} error"] = v * 0.07 * spread * np.exp(rng.normal(0, 0.25, n))
    for name, v in means.items():
        cols[f"worst {name}"] = v * (1 + 0.18 * spread * np.exp(rng.normal(0, 0.2, n)))
    X = pd.DataFrame(cols).round(5)
    return dict(X=X, y=y, target_names=np.array(["benign", "malignant"], dtype=object),
                source="SIMULATED tumour features (shaped like the Wisconsin diagnostic breast-cancer data)")


def simulate_blood_cells(seed: int = 1936) -> dict:
    """Synthetic flow-cytometry measurements of three white-blood-cell types (150 cells).

    50 lymphocytes, 50 monocytes and 50 granulocytes, each described by forward scatter
    (``FSC``, cell size), side scatter (``SSC``, granularity), ``CD45`` and ``CD14`` intensities
    (arbitrary log units).  Lymphocytes are easy to separate; monocytes and granulocytes
    overlap a little, so a classifier makes a few honest mistakes.

    Returns a dict with ``X`` (DataFrame, 150 × 4), ``y`` (0, 1, 2) and ``class_names``.
    """
    rng = np.random.default_rng(seed)
    centre = np.array([[3.0, 1.2, 4.6, 0.6],         # lymphocyte: small, agranular, CD45 bright, CD14−
                       [4.6, 2.6, 4.1, 2.9],         # monocyte: larger, CD14+
                       [4.8, 3.6, 3.6, 2.0]])        # granulocyte: granular (high SSC), CD45 dim
    scale = np.array([[0.30, 0.25, 0.20, 0.30],
                      [0.35, 0.35, 0.25, 0.45],
                      [0.35, 0.45, 0.25, 0.45]])
    y = np.repeat([0, 1, 2], 50)
    X = centre[y] + scale[y] * rng.normal(0, 1, (150, 4))
    X[:, 1] += 0.25 * (X[:, 0] - centre[y, 0])                           # size and granularity correlate
    return dict(X=pd.DataFrame(np.round(X, 3), columns=["FSC", "SSC", "CD45", "CD14"]), y=y,
                class_names=np.array(["lymphocyte", "monocyte", "granulocyte"], dtype=object))


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def train_test_indices(n: int, test_frac: float, rng: np.random.Generator, stratify=None):
    """Random train/test split of ``n`` samples; ``stratify`` keeps class proportions equal."""
    if stratify is None:
        perm = rng.permutation(n)
        n_test = int(round(test_frac * n))
        return np.sort(perm[n_test:]), np.sort(perm[:n_test])
    stratify = np.asarray(stratify)
    train, test = [], []
    for c in np.unique(stratify):
        idx = rng.permutation(np.flatnonzero(stratify == c))
        n_test = int(round(test_frac * len(idx)))
        test += list(idx[:n_test])
        train += list(idx[n_test:])
    return np.sort(train), np.sort(test)


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_decision_regions(predict, X, y, ax=None, resolution: int = 200, title: str = "",
                          class_names=None, pad: float = 0.5):
    """Shade the regions a 2-feature classifier assigns to each class and overlay the data.

    ``predict`` maps an (M, 2) array to M integer labels 0, 1, 2, …
    """
    X, y = np.asarray(X, float), np.asarray(y).astype(int)
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    gx, gy = np.meshgrid(np.linspace(x0, x1, resolution), np.linspace(y0, y1, resolution))
    Z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).astype(int).reshape(gx.shape)
    k = max(int(y.max()), int(Z.max())) + 1
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(CLASS_COLORS[:k])
    ax.contourf(gx, gy, Z, levels=np.arange(k + 1) - 0.5, cmap=cmap, alpha=0.18)
    ax.contour(gx, gy, Z, levels=np.arange(k) + 0.5, colors="k", linewidths=0.8)
    for c in range(k):
        m = y == c
        if m.any():
            ax.scatter(X[m, 0], X[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", linewidth=0.3,
                       label=(class_names[c] if class_names is not None else f"class {c}"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['load_play_tennis', 'simulate_tumor_features', 'simulate_blood_cells', 'simulate_genotypes', 'train_test_indices', 'accuracy', 'plot_decision_regions']))

---
## 1 · What a decision tree is

> 📍 **Where we are.** In [Module 08](08_knn_naive_bayes.ipynb) and [Module 09](09_logistic_regression.ipynb) you
> learned classifiers that are *formulas*: k-NN measures distances, naïve Bayes and logistic regression compute a
> probability. A decision tree is different — it is a **flowchart** of questions. This section shows what such a
> flowchart looks like and why it is useful; Sections 2–4 show how a computer *learns* one from data.

The lecture's running example is Tom Mitchell's **PlayTennis** table: on 14 days someone recorded the
weather — four **attributes** (features) *Outlook, Temperature, Humidity, Wind*, each taking a few discrete values —
and whether they played tennis (the **label** Y ∈ {Yes, No}). The goal is to learn a function
$f:\langle\text{Outlook, Temperature, Humidity, Wind}\rangle \to \text{PlayTennis}$.
The setting of the whole lecture: $Y$ is **discrete** (a class), the attributes are discrete but may also be
continuous (Section 7), and the approach is **non-probabilistic** — no likelihood, just a learned flowchart. The
lecture's three questions are *what decision trees are* (this section), *how to learn them* (Sections 2–4) and *how
to prune them* (Sections 5–6).

In [ ]:
# ── 1.1 The PlayTennis training data ──
tennis = load_play_tennis()
ATTRS = ["Outlook", "Temperature", "Humidity", "Wind"]
print(tennis.to_string(index=False))
print("\nvalues of each attribute:", {a: sorted(tennis[a].unique()) for a in ATTRS})
print("label counts:", tennis["PlayTennis"].value_counts().to_dict())

Below is the tree that the learning algorithm will produce (we draw it in cell 1.3). Its vocabulary:

| Part | What it does | In the PlayTennis tree |
|---|---|---|
| **internal node** | tests one attribute $X_i$ | *Outlook*, *Humidity*, *Wind* |
| **branch** | one value of the tested attribute | *Sunny*, *Overcast*, *Rain*; *High*, *Normal*; *Strong*, *Weak* |
| **leaf** | predicts Y (or the class frequencies $P(Y\mid X\in\text{leaf})$) | *Yes* / *No* |
| **root** | the first internal node | *Outlook* |

To classify a new day you start at the root and follow the branch matching that day's value until you reach a leaf.
Here is that tree written by hand as a Python function, so you can see that a tree *is* just nested IF statements.

In [ ]:
# ── 1.2 The PlayTennis tree, written by hand ──
def tennis_tree(day: dict) -> str:
    """The PlayTennis decision tree as nested if/else statements."""
    if day["Outlook"] == "Overcast":
        return "Yes"
    if day["Outlook"] == "Sunny":
        return "No" if day["Humidity"] == "High" else "Yes"
    return "No" if day["Wind"] == "Strong" else "Yes"          # Outlook == "Rain"


pred = np.array([tennis_tree(row) for row in tennis.to_dict("records")])
print("predictions:", pred)
print(f"training accuracy of the hand-written tree: {accuracy(tennis['PlayTennis'], pred):.2f}")
print("D2 (Sunny, Hot, High, Strong) →", tennis_tree(tennis.iloc[1].to_dict()),
      "  path: Outlook = Sunny → Humidity = High → leaf 'No'")

A tree is easier to read as a picture. The function below draws any tree stored as nested dictionaries — the format
our learning algorithm of Section 4 will produce: an internal node holds the index of the tested attribute
(`"feature"`) and a dictionary `"children"` from attribute values to subtrees; every node remembers its majority
label (`"majority"`) and number of training examples (`"n"`). We type the PlayTennis tree in that format by hand.

In [ ]:
# ── 1.3 Draw a decision tree with Matplotlib ──
def draw_tree(node: dict, names, ax=None, title: str = "", positive="Yes", fontsize: int = 8) -> None:
    """Draw a nested-dict tree: blue boxes = attribute tests, green/red ovals = leaves (label and size)."""
    ax = ax if ax is not None else plt.gca()
    pos, n_leaves = {}, [0]

    def layout(nd, depth):                                    # leaves get x = 0, 1, 2, …; parents sit above
        if "feature" not in nd:
            pos[id(nd)] = (n_leaves[0], -depth)
            n_leaves[0] += 1
        else:
            xs = [layout(c, depth + 1)[0] for c in nd["children"].values()]
            pos[id(nd)] = (float(np.mean(xs)), -depth)
        return pos[id(nd)]

    def draw(nd):
        x, yv = pos[id(nd)]
        if "feature" not in nd:
            colour = "#c7e9c0" if nd["majority"] == positive else "#fcbba1"
            ax.text(x, yv, f"{nd['majority']}\n({nd['n']})", ha="center", va="center", fontsize=fontsize, zorder=3,
                    bbox=dict(boxstyle="round,pad=0.3", fc=colour, ec="0.4"))
            return
        for value, child in nd["children"].items():
            cx, cy = pos[id(child)]
            ax.plot([x, cx], [yv, cy], color="0.55", lw=1, zorder=1)
            ax.text((x + cx) / 2, (yv + cy) / 2, str(value), ha="center", va="center", fontsize=fontsize - 1,
                    color="0.25", zorder=2, bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none"))
            draw(child)
        ax.text(x, yv, names[nd["feature"]], ha="center", va="center", fontsize=fontsize, zorder=3,
                bbox=dict(boxstyle="square,pad=0.35", fc="#deebf7", ec="#3182bd"))

    _, _ = layout(node, 0)
    draw(node)
    depth = -min(p[1] for p in pos.values())
    ax.set_xlim(-0.7, n_leaves[0] - 0.3); ax.set_ylim(-depth - 0.5, 0.5)
    ax.axis("off"); ax.set_title(title)


leaf = lambda label, n: {"majority": label, "n": n}
TREE_BY_HAND = {"feature": 0, "majority": "Yes", "n": 14, "children": {          # 0 = Outlook
    "Sunny": {"feature": 2, "majority": "No", "n": 5,                             # 2 = Humidity
              "children": {"High": leaf("No", 3), "Normal": leaf("Yes", 2)}},
    "Overcast": leaf("Yes", 4),
    "Rain": {"feature": 3, "majority": "Yes", "n": 5,                             # 3 = Wind
             "children": {"Strong": leaf("No", 2), "Weak": leaf("Yes", 3)}}}}

fig, ax = plt.subplots(figsize=(6.0, 3.0))
draw_tree(TREE_BY_HAND, ATTRS, ax=ax, title="the PlayTennis decision tree (number of training days in brackets)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The three-question tree classifies all 14 training days correctly; the diagram shows the
same nested IF statements as boxes (tests), labelled branches (values) and leaves (predictions). Day D2 travels
Sunny → High → *No*. Notice what the tree does *not* use: **Temperature never appears**, and on an overcast day you
need only one question. The lecture lists why this matters:

- **Small and readable.** A clinician can check a three-question tree; nobody can check 3,571 regression weights.
- **Interactions.** "Sunny *and* humid → no tennis" is an **interaction**: the effect of Humidity depends on Outlook
  (on rainy days humidity is irrelevant). A linear classifier (Module 09) adds up feature effects and cannot express
  "Humidity matters only if Sunny" without hand-made product features. In genetics this is **epistasis** — the effect
  of one locus depends on the genotype at another — and we will simulate exactly that in Section 5.
- **Cheap at test time.** You evaluate only the attributes on your path. When features have different **costs** —
  "blood pressure < 100?" is free, "MRI scan normal?" is expensive — you can even prefer trees that ask the cheap
  questions first.

- **Insight.** Often a fairly accurate classifier is also small; reading it can teach you something about the problem
  or reveal a bug in your feature set (a tree that splits first on "sample batch" is telling you something).

✏️ **Your turn.** Classify the day (Rain, Hot, High, Weak) with the tree — first by hand from the diagram, then with
`tennis_tree({"Outlook": "Rain", "Temperature": "Hot", "Humidity": "High", "Wind": "Weak"})`. Which attributes did you
have to look at, and which did you never need?

### 📘 Lecture notes — what a decision tree is

**Setting.** Learn a classifier $f: X \to Y$ where $Y$ is discrete and the attributes $X_1, \dots, X_J$ are
discrete (or continuous, split by thresholds). Decision trees are a **non-probabilistic** approach: no
distribution is modelled, a flowchart is learned.

**Anatomy.**
- each **internal node** tests one attribute $X_i$;
- each **branch** from a node selects one value of $X_i$ (or one side of a threshold);
- each **leaf** predicts $Y$ — or the class frequencies $P(Y \mid X \in \text{leaf})$;
- to classify, start at the **root** and follow the branches that match the example until a leaf is reached.

**Worked example (PlayTennis, 14 days, 9 Yes / 5 No).** Root *Outlook*: Overcast → *Yes* (4 days);
Sunny → test *Humidity* (High → *No*, Normal → *Yes*); Rain → test *Wind* (Strong → *No*, Weak → *Yes*).
Day D2 = (Sunny, Hot, High, Strong) follows Sunny → High → *No*. The tree fits all 14 days and never uses
*Temperature*.

**Why trees.** (i) Often a small, easily understood classifier is fairly accurate, and reading it gives insight
or helps debug the features. (ii) Trees capture **interactions** ("sunny *and* humid") that a linear classifier
cannot express without hand-made product features — in genetics, **epistasis**. (iii) They are cheap at test time:
only the attributes on one path are measured, and features with different **costs** ("blood pressure < 100?" vs
"MRI scan normal?") can be taken into account.

**Common confusion.** A tree's leaves can output probabilities (class frequencies), but the tree is still not a
probabilistic *model* of the data: it is grown greedily by an impurity criterion, not by maximising a likelihood.

> 🎓 **Exam tip.** Be able to classify a new example by hand from a drawn tree, and to explain why a tree can represent an
    interaction (an XOR-like pattern) that a linear decision boundary cannot.

---
## 2 · Entropy: measuring how mixed a set of labels is

> 📍 **Where we are.** We know what a tree is. To *learn* one, the computer must decide which question to ask first.
> Most tree-learning algorithms (ID3, C4.5, CART) follow the same generic recipe:

```text
grow_tree(D):
    if all examples in D have the same label y (or nearly so):  return leaf(y)
    pick the best attribute a and split D into D1, D2, …, Dk  (a = c1, a = c2, … ; or a < θ vs a ≥ θ)
    for each Dk:  child_k = grow_tree(Dk)                      ← the same procedure, called on a smaller set
```

This is **greedy** (it picks the best question *now* and never reconsiders) and **top-down** (root first). The
open question is what "best" means, and the training data must decide it. The popular answer: prefer splits whose
subsets are **pure** — each contains few different labels, or a very skewed distribution of labels. We need a number
that measures impurity: **entropy**.

🧮 **Math refresher — logarithms base 2.** $\log_2 x$ answers "2 to what power gives $x$?": $\log_2 8 = 3$ because
$2^3 = 8$; $\log_2 1 = 0$; $\log_2 \tfrac12 = -1$; $\log_2 \tfrac14 = -2$. For numbers between 0 and 1 the log is
negative. NumPy writes it `np.log2`. (The course's `log` without a base is the natural log; entropies always use
$\log_2$ and are measured in **bits**.) The symbol $\sum_{i=1}^{n}$ means "add up the following term for
$i = 1, 2, \dots, n$".

Let $S$ be a set of training examples, $p_\oplus$ the fraction that is positive (*Yes*) and $p_\ominus$ the fraction
that is negative (*No*). The **sample entropy** of the labels is

$$H_S(Y) = -\,p_\oplus \log_2 p_\oplus \;-\; p_\ominus \log_2 p_\ominus ,$$

*In words:* weigh each class's "surprise" $-\log_2 p$ (large when the class is rare) by how often the class occurs,
and add. A pure set (all *Yes*) has entropy 0 — no surprise at all; a 50/50 set has entropy 1 bit, the maximum for
two classes. By convention $0\log_2 0 = 0$ (a class that never occurs contributes nothing).

Two worked cases: a set of 16 examples that are **all positive** has $p_\oplus = 1, p_\ominus = 0$, so
$H = -1\log_2 1 - 0\log_2 0 = 0$; a set that is **half positive, half negative** has
$H = -\tfrac12\log_2\tfrac12 - \tfrac12\log_2\tfrac12 = -\tfrac12(-1) - \tfrac12(-1) = 1$.

With more than two classes the same idea reads

$$H(X) = -\sum_{i=1}^{n} P(X=i)\,\log_2 P(X=i),$$

*In words:* the average surprise over the $n$ possible values of $X$; its maximum is $\log_2 n$ bits, reached when all
$n$ values are equally likely.

In [ ]:
# ── 2.1 Entropy of a set of labels ──
def entropy_bits(labels) -> float:
    """Sample entropy (in bits) of a 1-D array of class labels; 0·log2(0) is treated as 0."""
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)       # how many examples of each class
    p = counts / counts.sum()                                # class proportions (none is 0 here)
    return float(-(p * np.log2(p)).sum()) + 0.0              # "+ 0.0" turns −0.0 into 0.0


print("all positive  (16+, 0−):", entropy_bits(["+"] * 16))                       # pure set: 0 bits
print("half and half ( 7+, 7−):", entropy_bits(["+"] * 7 + ["-"] * 7))             # maximally mixed: 1 bit
print("PlayTennis    ( 9+, 5−):", round(entropy_bits(tennis["PlayTennis"]), 4))    # the root of PlayTennis
print("four equally likely DNA bases:", entropy_bits(list("ACGT")), "bits = log2(4)")

In [ ]:
# ── 2.2 The entropy curve for a binary label ──
p = np.linspace(0, 1, 401)
with np.errstate(divide="ignore", invalid="ignore"):
    H = -np.nan_to_num(p * np.log2(p)) - np.nan_to_num((1 - p) * np.log2(1 - p))

fig, ax = plt.subplots(figsize=(5.2, 3.3))
ax.plot(p, H, color="tab:blue", lw=2)
for pp, lab in [(0, "pure: 0 bits"), (0.5, "50/50: 1 bit"), (9 / 14, "PlayTennis 9+/5−")]:
    hh = entropy_bits([1] * int(round(pp * 1000)) + [0] * int(round((1 - pp) * 1000)))
    ax.plot(pp, hh, "o", color="tab:red")
    offset = {0: (8, 6), 0.5: (-30, -38)}.get(pp, (8, 6))
    ax.annotate(f"{lab}\nH = {hh:.3f}", (pp, hh), textcoords="offset points", xytext=offset, fontsize=8)
ax.set_xlabel(r"fraction of positive examples $p_\oplus$")
ax.set_ylabel(r"entropy $H_S(Y)$ (bits)")
ax.set_ylim(-0.03, 1.18)                                      # head-room for the labels
ax.set_title("entropy measures the impurity of a set of labels")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The function reproduces the hand calculations: 0 bits for an all-positive set, exactly
1 bit for a 50/50 set, and **0.940 bits** for the 9 *Yes* / 5 *No* PlayTennis days. The curve is symmetric (swapping
the labels does not change how mixed a set is) and peaks at $p_\oplus = 0.5$.

### 2.3 Why "bits"? The coding interpretation

Information theory gives entropy a concrete meaning: $H(X)$ is *the expected number of bits needed to encode a
randomly drawn value of $X$ under the most efficient code*, because the most efficient code spends $-\log_2 P(X=i)$
bits on value $i$ — so the expected code length is $\sum_i P(X=i)\,(-\log_2 P(X=i)) = H(X)$. Here is a biological
example. Suppose a genome region has a skewed base composition: A with probability 1/2, T 1/4, C 1/8 and G 1/8.
Instead of the obvious 2 bits per base, use a **prefix code** with short codewords for frequent bases:
A = `0`, T = `10`, C = `110`, G = `111` (no codeword is the start of another, so a bit string decodes uniquely).
The codeword lengths 1, 2, 3, 3 are exactly $-\log_2(1/2), -\log_2(1/4), -\log_2(1/8), -\log_2(1/8)$.

In [ ]:
# ── 2.3 Entropy = average code length of the best code ──
probs = {"A": 1 / 2, "T": 1 / 4, "C": 1 / 8, "G": 1 / 8}
codebook = {"A": "0", "T": "10", "C": "110", "G": "111"}
H_bases = -sum(q * np.log2(q) for q in probs.values())
seq = rng.choice(list(probs), size=100_000, p=list(probs.values()))      # a long random 'sequence'
bits = "".join(codebook[b] for b in seq[:20])
avg_len = np.mean([len(codebook[b]) for b in seq])
print("first 20 bases :", "".join(seq[:20]))
print("encoded        :", bits)
print(f"entropy of the base distribution: {H_bases:.3f} bits/base")
print(f"average code length on 100,000 bases: {avg_len:.3f} bits/base   (naive fixed code: 2 bits/base)")

🔍 **What just happened?** The variable-length code needs on average about 1.75 bits per base — the entropy — instead
of 2. A low-entropy (predictable) source is cheap to describe; a high-entropy one is expensive. For tree learning
this means: **a pure node needs no further questions (0 bits), a mixed node still needs about one yes/no answer per
example.** A good split is one that removes many bits of uncertainty.

✏️ **Your turn.** Change `probs` to the uniform composition (1/4 each) and keep the same `codebook`. What is the new
entropy, what average code length does the (now mismatched) code achieve, and why is it worse than 2 bits?

---
## 3 · Conditional entropy and information gain: choosing the question

> 📍 **Where we are.** Entropy scores one set of labels. A split creates *several* subsets; we score the split by
> their average entropy and compare it with the entropy before the split.

Splitting $S$ on attribute $A$ sends the examples with $A = v$ into the subset $S_v$. We define

$$H(Y\mid A=v) = -\sum_{y} P(Y=y\mid A=v)\log_2 P(Y=y\mid A=v), \qquad
  H(Y\mid A) = \sum_{v\in\text{values}(A)} P(A=v)\, H(Y\mid A=v),$$

*In words:* the **specific conditional entropy** $H(Y\mid A=v)$ is the label entropy inside branch $v$; the
**conditional entropy** $H(Y\mid A)$ averages those branch entropies, weighting each branch by the fraction of
examples that go there, $P(A=v) = |S_v|/|S|$. The **information gain** is the drop in entropy:

$$\operatorname{Gain}(S, A) = H_S(Y) - H_S(Y\mid A) = I_S(A, Y).$$

*In words:* how many bits of uncertainty about the label disappear, on average, once we know $A$ — the **expected
reduction in entropy** of $Y$ caused by sorting the sample on $A$. This quantity is the **mutual information**
between the attribute and the label (information theory writes the same formulas for any two random variables $X$
and $Y$). The greedy rule: **split on the attribute with the largest gain.**

Let us work through *Humidity* step by step: High → [3+, 4−], Normal → [6+, 1−].

In [ ]:
# ── 3.1 Information gain of Humidity, by hand ──
H_S = -(9 / 14) * np.log2(9 / 14) - (5 / 14) * np.log2(5 / 14)
H_high = -(3 / 7) * np.log2(3 / 7) - (4 / 7) * np.log2(4 / 7)
H_normal = -(6 / 7) * np.log2(6 / 7) - (1 / 7) * np.log2(1 / 7)
H_cond = (7 / 14) * H_high + (7 / 14) * H_normal
print(f"H_S(Y)                 = {H_S:.3f}")
print(f"H(Y | Humidity=High)   = {H_high:.3f}     [3+, 4−]")
print(f"H(Y | Humidity=Normal) = {H_normal:.3f}     [6+, 1−]")
print(f"H(Y | Humidity)        = 7/14·{H_high:.3f} + 7/14·{H_normal:.3f} = {H_cond:.3f}")
print(f"Gain(S, Humidity)      = {H_S:.3f} − {H_cond:.3f} = {H_S - H_cond:.4f}")

In [ ]:
# ── 3.2 Information gain of every attribute at the root ──
def information_gain(x, y) -> float:
    """Gain = H(Y) − H(Y | X) for a discrete attribute x and labels y (arrays of equal length)."""
    x, y = np.asarray(x), np.asarray(y)
    h_cond = sum(np.mean(x == v) * entropy_bits(y[x == v]) for v in np.unique(x))
    return entropy_bits(y) - h_cond


def gain_rounded(x, y, decimals: int = 3) -> float:
    """The same gain, but rounding every intermediate entropy to 3 decimals as in Mitchell's textbook."""
    x, y = np.asarray(x), np.asarray(y)
    r = lambda h: round(h, decimals)
    return r(entropy_bits(y)) - sum(np.mean(x == v) * r(entropy_bits(y[x == v])) for v in np.unique(x))


y_tennis = tennis["PlayTennis"].to_numpy()
table = pd.DataFrame({a: {"exact gain (bits)": information_gain(tennis[a], y_tennis),
                          "rounded-intermediates gain": gain_rounded(tennis[a], y_tennis)} for a in ATTRS}).T
print(table.sort_values("exact gain (bits)", ascending=False).round(4).to_string())

In [ ]:
# ── 3.2b Humidity vs Wind: the two candidate splits as diagrams ──
def draw_split(ax, attr: str) -> None:
    """Draw the root [9+, 5−] split on one attribute with the entropy of every node and the gain."""
    ax.axis("off"); ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.45, 0.55)
    root = f"S: [{(y_tennis == 'Yes').sum()}+, {(y_tennis == 'No').sum()}−]\nH = {entropy_bits(y_tennis):.3f}"
    ax.text(0, 0.15, root, ha="center", va="center", fontsize=8, bbox=dict(boxstyle="round", fc="#deebf7", ec="#3182bd"))
    values = sorted(tennis[attr].unique())
    xs = np.linspace(-0.85, 0.85, len(values))
    h_cond = 0.0
    for x, v in zip(xs, values):
        yv = y_tennis[tennis[attr] == v]
        h_cond += len(yv) / len(y_tennis) * entropy_bits(yv)
        ax.annotate("", xy=(x, -0.72), xytext=(0, -0.1), arrowprops=dict(arrowstyle="-", color="0.5"))
        ax.text(x / 2, -0.38, v, ha="center", fontsize=7, color="0.25", bbox=dict(fc="white", ec="none", pad=0.5))
        ax.text(x, -0.95, f"[{(yv == 'Yes').sum()}+, {(yv == 'No').sum()}−]\nH = {entropy_bits(yv):.3f}", ha="center",
                va="center", fontsize=8, bbox=dict(boxstyle="round", fc="#f0f0f0", ec="0.5"))
    ax.set_title(f"split on {attr}: Gain = {entropy_bits(y_tennis):.3f} − {h_cond:.3f} = "
                 f"{entropy_bits(y_tennis) - h_cond:.3f} bits", fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(8.6, 2.8))
draw_split(axes[0], "Humidity"); draw_split(axes[1], "Wind")
plt.suptitle("which attribute should we split on? the one with the larger information gain", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** *Outlook* has the largest gain (0.247 bits), so it becomes the **root** — exactly the tree
drawn in Section 1. *Humidity* (0.152) beats *Wind* (0.048): the diagrams show why — Humidity produces one fairly
pure branch ([6+, 1−], 0.592 bits), whereas Wind leaves one branch completely mixed ([3+, 3−], 1 bit). *Temperature*
(0.029) is nearly useless at the root.

⚠️ **Rounding.** Mitchell's textbook quotes Humidity 0.151 and Outlook 0.246: it plugs the rounded entropies 0.940,
0.985, 0.592, 0.971 into the formula and keeps three decimals. The second column follows that recipe (0.2464, 0.1515,
0.0480, 0.0291, which Mitchell writes as 0.246, 0.151, 0.048, 0.029); the exact values are 0.247, 0.152, 0.048, 0.029
(the hand calculation of 3.1 gives 0.940 − (7/14)·0.985 − (7/14)·0.592 = 0.940 − 0.788 = 0.152). Either way the
ranking — and the tree — is the same, so do not worry if an exam answer differs in the third decimal.

### 3.3 Information gain is mutual information — and it is symmetric

Mutual information $I(A, Y) = H(Y) - H(Y\mid A)$ can also be written $H(A) - H(A\mid Y)$: knowing the label tells you as
much about the attribute as the attribute tells you about the label. Let us check that numerically — a good habit
whenever a textbook states an identity.

In [ ]:
# ── 3.3 Check the symmetry I(A, Y) = I(Y, A) ──
for a in ATTRS:
    x = tennis[a].to_numpy()
    print(f"{a:12s}  H(Y) − H(Y|A) = {information_gain(x, y_tennis):.4f}    H(A) − H(A|Y) = {information_gain(y_tennis, x):.4f}")

⚠️ **A trap: attributes with many values.** Information gain favours attributes that split the data into many small
pure pieces. The *Day* column (D1 … D14) is unique for every example, so splitting on it creates 14 pure leaves.

In [ ]:
# ── 3.4 The 'Day' attribute has maximal gain — and is useless ──
print(f"Gain(S, Day) = {information_gain(tennis['Day'], y_tennis):.3f} bits  (= H_S(Y), the largest possible)")
split_info = entropy_bits(tennis["Day"])           # entropy of the attribute itself: log2(14) bits
print(f"gain ratio (C4.5) of Day     = {information_gain(tennis['Day'], y_tennis) / split_info:.3f}")
print(f"gain ratio (C4.5) of Outlook = {information_gain(tennis['Outlook'], y_tennis) / entropy_bits(tennis['Outlook']):.3f}")

🔍 **What just happened?** *Day* removes all 0.940 bits of uncertainty on the training data but cannot predict
anything about a new day — the purest possible overfitting. Quinlan's C4.5 reduces the bias with the **gain ratio**,
which divides the gain by the attribute's own entropy $H(A)$ ($\log_2 14 = 3.81$ bits for *Day*, 1.58 bits for
*Outlook*). It helps — *Day*'s score drops from 0.940 (gain) to 0.247 (gain ratio) — but on 14 examples *Day* still
beats *Outlook* (gain ratio 0.156)!
A correction factor cannot rescue an attribute that is an identifier. The real fix is to **never use IDs (patient
numbers, sample barcodes, plate positions) as features.** The same bias towards attributes with many distinct values
returns for continuous features in Section 10.

✏️ **Your turn.** Restrict the table to the five *Sunny* days (`tennis[tennis.Outlook == "Sunny"]`) and compute the
gain of Humidity, Temperature and Wind. Mitchell's textbook reports 0.970, 0.570 and 0.019 (rounded intermediates).
Which attribute should be tested under the *Sunny* branch?

### 📘 Lecture notes — entropy, conditional entropy and information gain

**Greedy top-down learning.** Given data $D$: if all (or nearly all) examples have the same label $y$, return a
leaf $y$; otherwise pick the best split on the best attribute $a$ — $a$ vs not $a$, $a = c_1$ or $a = c_2$ or …,
or $a < \theta$ vs $a \ge \theta$ — split $D$ into $D_1, \dots, D_k$ and build a tree for each $D_k$ recursively.
"Best" = the split that most lowers the **entropy** of the labels.

**Sample entropy** of a set $S$ with fractions $p_\oplus, p_\ominus$ of positive and negative examples:

$$H_S(Y) = -p_\oplus\log_2 p_\oplus - p_\ominus\log_2 p_\ominus .$$

*In words:* 0 bits for a pure set (16+, 0−), 1 bit for a 50/50 set, in between otherwise ($0\log_2 0 = 0$). In
general $H(X) = -\sum_{i=1}^{n} P(X=i)\log_2 P(X=i)$, the expected number of bits of the most efficient code,
at most $\log_2 n$.

**Conditional entropy and information gain.**

$$H(Y\mid A=v) = -\sum_y P(y\mid A=v)\log_2 P(y\mid A=v),\quad H(Y\mid A) = \sum_v P(A=v)\,H(Y\mid A=v),\quad
\operatorname{Gain}(S,A) = H_S(Y) - H_S(Y\mid A) = I(A, Y).$$

*In words:* average the branch entropies, weighted by the branch sizes, and subtract from the entropy before the
split; the gain is the mutual information between attribute and label, the expected reduction in label entropy
from sorting on $A$.

**Worked example (PlayTennis root, [9+, 5−], $H = 0.940$).**

| attribute | branches | $H(Y\mid A)$ | gain |
|---|---|---|---|
| Humidity | High [3+, 4−] 0.985 · Normal [6+, 1−] 0.592 | (7/14)·0.985 + (7/14)·0.592 = 0.788 | **0.152** |
| Wind | Weak [6+, 2−] 0.811 · Strong [3+, 3−] 1.000 | (8/14)·0.811 + (6/14)·1.000 = 0.892 | 0.048 |
| Outlook | Sunny [2+, 3−] · Overcast [4+, 0−] · Rain [3+, 2−] | 0.694 | **0.247** (root) |
| Temperature | Hot [2+, 2−] · Mild [4+, 2−] · Cool [3+, 1−] | 0.911 | 0.029 |

Under *Sunny* ([2+, 3−], 0.971 bits) the gains are Humidity 0.971, Temperature 0.571, Wind 0.020 → test Humidity.
(Textbooks that round intermediate entropies quote 0.151, 0.246, 0.970, 0.570, 0.019.)

**Common confusions.** Forgetting to weight branch entropies by $|S_v|/|S|$; using natural logs (entropies here
are in bits, $\log_2$); and trusting a large gain for an identifier-like attribute (every value pure → gain =
$H(Y)$, but no predictive value; C4.5's gain ratio divides by $H(A)$).

> 🎓 **Exam tip.** Practise computing H(Y), H(Y|A) and Gain(S, A) from a small table by hand — it is a classic exam question.
    Remember to weight each branch entropy by the fraction of examples in that branch, and that −0·log₂0 = 0.

---
## 4 · ID3 from scratch

> 📍 **Where we are.** We can pick the best attribute for one node. **ID3** (Iterative Dichotomiser 3; Quinlan 1986)
> simply applies that choice recursively, exactly as in the recipe of Section 2.

🧮 **Programming refresher — recursion.** A *recursive* function calls itself on a smaller problem. `id3(S)` builds the
root of the tree for the examples $S$, and for each branch calls `id3(S_v)` on the subset — which builds that subtree's
root and calls itself again — until a subset is pure. Every call needs a **stopping rule** (a *base case*); otherwise
it would never finish. ID3 stops when (i) all labels are equal, (ii) no attributes are left (an attribute is used at
most once on any path), or (iii) no split has positive gain. A leaf predicts the **majority label** of its examples.

We store a tree as nested Python dictionaries: an internal node holds the tested attribute and a dictionary
`children` from attribute values to subtrees; every node also remembers its majority label and size, which we need
for prediction on unseen values and for pruning later.

In [ ]:
# ── 4.1 The ID3 algorithm ──
def majority_label(y):
    """Most frequent label (ties broken by sorted order)."""
    values, counts = np.unique(y, return_counts=True)
    return values[np.argmax(counts)]


def id3(X, y, features, depth: int = 0, max_depth=None) -> dict:
    """Grow an ID3 tree on discrete attributes.

    X: (N, J) array of attribute values; y: (N,) labels; features: list of column indices still available.
    Returns a nested dict; internal nodes have keys 'feature', 'gain' and 'children'.
    """
    node = {"majority": majority_label(y), "n": len(y)}
    if len(np.unique(y)) == 1 or not features or (max_depth is not None and depth >= max_depth):
        return node                                            # base cases → leaf
    gains = [information_gain(X[:, f], y) for f in features]
    if max(gains) <= 0:
        return node                                            # no attribute helps → leaf
    best = features[int(np.argmax(gains))]
    node.update(feature=best, gain=max(gains), children={})
    remaining = [f for f in features if f != best]             # each attribute at most once per path
    for v in np.unique(X[:, best]):
        mask = X[:, best] == v
        node["children"][v] = id3(X[mask], y[mask], remaining, depth + 1, max_depth)
    return node


def print_tree(node: dict, names, indent: str = "") -> None:
    """Pretty-print a tree grown by id3()."""
    if "feature" not in node:
        print(f"{indent}→ {node['majority']}  ({node['n']} examples)")
        return
    print(f"{indent}[{names[node['feature']]}]  gain = {node['gain']:.3f} bits")
    for v, child in node["children"].items():
        print(f"{indent}   {names[node['feature']]} = {v}:")
        print_tree(child, names, indent + "      ")


X_tennis = tennis[ATTRS].to_numpy()
tree_tennis = id3(X_tennis, y_tennis, features=list(range(len(ATTRS))))
print_tree(tree_tennis, ATTRS)

In [ ]:
# ── 4.2 Predicting with a learned tree ──
def predict_row(node: dict, x):
    """Follow the branches matching x; an unseen value stops at the current node's majority label."""
    while "feature" in node:
        child = node["children"].get(x[node["feature"]])
        if child is None:
            break
        node = child
    return node["majority"]


def predict_tree(node: dict, X) -> np.ndarray:
    """Predict every row of X."""
    return np.array([predict_row(node, x) for x in X])


def count_nodes(node: dict) -> int:
    """Number of nodes (internal + leaves)."""
    return 1 + sum(count_nodes(c) for c in node.get("children", {}).values())


print("training accuracy:", accuracy(y_tennis, predict_tree(tree_tennis, X_tennis)))
print("number of nodes  :", count_nodes(tree_tennis))
new_day = np.array(["Sunny", "Cool", "High", "Strong"])        # Mitchell's classic test day
print("new day", dict(zip(ATTRS, new_day.tolist())), "→", predict_row(tree_tennis, new_day))
print("learned tree identical to the hand-written one:", tree_tennis["children"].keys() == TREE_BY_HAND["children"].keys()
      and all(predict_row(tree_tennis, d) == tennis_tree(dict(zip(ATTRS, d))) for d in X_tennis))

🔍 **What just happened?** Our ID3 rediscovered the tree of Section 1: *Outlook* at the root, *Humidity* under *Sunny*
(gain 0.971 bits — it makes both branches pure), *Wind* under *Rain*, and an immediate *Yes* leaf for *Overcast*.
The tree has 8 nodes (3 tests, 5 leaves) and fits all 14 days. The exact Sunny-branch gains are 0.971 (Humidity),
0.571 (Temperature) and 0.020 (Wind); the textbook values 0.970/0.570/0.019 again use rounded intermediates.

⚠️ **ID3 is greedy.** It never backtracks. A sequence of locally best questions need not give the smallest or most
accurate tree (finding the smallest consistent tree is computationally intractable in general). The consequence,
listed among the cons of trees in Section 8: an error at the top of the tree affects the rest of the tree-growing
process.

### 4.3 Trees are IF-THEN rules

Each path from the root to a leaf is one rule: the tests along the path joined by AND, the leaf as the conclusion.
Rules are often easier to discuss with collaborators, and they can be simplified afterwards.

In [ ]:
# ── 4.3 One IF-THEN rule per leaf ──
def tree_to_rules(node: dict, names, conditions=()) -> list:
    """List of (conditions, prediction) pairs, one per leaf."""
    if "feature" not in node:
        return [(conditions, node["majority"])]
    rules = []
    for v, child in node["children"].items():
        rules += tree_to_rules(child, names, conditions + (f"{names[node['feature']]} = {v}",))
    return rules


for conds, label in tree_to_rules(tree_tennis, ATTRS):
    print("IF", " AND ".join(conds), "THEN PlayTennis =", label)


def simplified_rules(day) -> str:
    """A shorter rule set: two 'No' rules and a default."""
    outlook, temp, humidity, wind = day
    if outlook == "Sunny" and humidity == "High":
        return "No"
    if outlook == "Rain" and wind == "Strong":
        return "No"
    return "Yes"


all_days = np.array(np.meshgrid(*[sorted(tennis[a].unique()) for a in ATTRS])).reshape(4, -1).T
same = np.mean([simplified_rules(d) == predict_row(tree_tennis, d) for d in all_days])
print(f"\nsimplified rules agree with the tree on {same:.0%} of all {len(all_days)} possible days")

🔍 **What just happened?** Five leaves give five rules — one rule per leaf. Because the rules are tried in order,
they can be collapsed into a simpler rule set: "IF Sunny AND humid THEN no; ELSE IF Rain AND windy THEN no; ELSE
play" — and the code confirms that this shorter list makes the same decision on all 36 possible weather combinations.
(With a version of the data in which humidity is *numeric*, a C4.5 tree splits the sunny branch on humidity at a
threshold between the observed values 70 and 85 — the rule becomes "IF Sunny AND humidity above the threshold THEN
no"; we come to numeric attributes in Section 7.)

✏️ **Your turn.** Remove *Outlook* from the feature list (`features=[1, 2, 3]`) and grow the tree again. Which
attribute becomes the root, how many nodes does the tree need, and does it still fit all 14 days?

---
## 5 · Noise and overfitting

> 📍 **Where we are.** ID3 keeps splitting until every leaf is pure. That is perfect for clean data — but biological
> labels are noisy (mis-annotated samples, incomplete penetrance, technical artefacts). What does one wrong label do
> to the tree?

The lecture adds a **noisy** training day: **D15 = (Sunny, Hot, Normal, Strong) → No**. According to the tree we just
learned, a sunny day with normal humidity means *Yes* — D15 lands in the *Sunny → Normal → Yes* leaf with the wrong
label. So either D15 reveals a real exception, or it is a recording error.

In [ ]:
# ── 5.1 Add the noisy day D15 ──
tennis15 = load_play_tennis(noisy=True)
X15, y15 = tennis15[ATTRS].to_numpy(), tennis15["PlayTennis"].to_numpy()
tree15 = id3(X15, y15, features=list(range(len(ATTRS))))
print_tree(tree15, ATTRS)
print(f"\nnodes: {count_nodes(tree_tennis)} without D15 → {count_nodes(tree15)} with D15")
sunny = tennis15["Outlook"] == "Sunny"
print("gains at the Sunny node with D15:",
      {a: round(float(information_gain(tennis15.loc[sunny, a], y15[sunny])), 3)
       for a in ["Humidity", "Temperature", "Wind"]})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.3), gridspec_kw={"width_ratios": [5, 7]})
draw_tree(tree_tennis, ATTRS, ax=axes[0], title="14 days: 8 nodes")
draw_tree(tree15, ATTRS, ax=axes[1], title="with the noisy day D15: 11 nodes", fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** One mislabelled day makes the tree grow from 8 to 11 nodes — and it does more than hang an
extra test below the *Humidity = Normal* leaf where D15 lands in the original tree: with D15 the Sunny subset is
[2+, 4−], *Temperature* now has the higher gain at that node (0.585 vs 0.459 bits), so the **tested attribute itself
changes** and Humidity is pushed one level down under *Mild*. The tree now encodes the rule "Sunny and Hot → No",
which is true of the three hot sunny days in the table but was "learned" largely to accommodate one possibly wrong
label.

The dilemma: *do we want to be accurate on D15?* If D15 is noise we would rather keep the small tree; if it is real
we want the big one — and **we usually do not know ahead of time which**. The danger has a precise name:
**overfitting** occurs when the error on the training data is low but the error on *all* data (the population we
care about) is high. The remedy: split the available data into training and test data, and measure error on data
the learner never saw.

### 5.2 A simulated genetic study with epistasis and label noise

To see overfitting as a curve — accuracy against tree size, the classic picture from Mitchell's book (there drawn
for a medical-diagnosis data set) — we need more data. We simulate a case–control study:
600 individuals genotyped at 10 SNPs (genotypes 0/1/2 = copies of the risk allele, as in Modules 06 and 08). The
**true** disease rule contains an **epistatic interaction** and a recessive locus:

$$\text{case} \iff (\text{SNP1} \ge 1 \;\text{AND}\; \text{SNP2} \ge 1) \;\text{OR}\; \text{SNP3} = 2 ,$$

*In words:* you are affected if you carry at least one risk allele at *both* SNP1 and SNP2, or two risk alleles at
SNP3. To mimic phenotyping errors and incomplete penetrance, **15 % of labels are flipped at random**. SNPs 4–10 are
pure noise. We split the individuals into 300 training, 150 validation (used in Section 6) and 150 test.

In [ ]:
# ── 5.2 Simulate the case–control data (SIMULATED) ──
rng = np.random.default_rng(SEED + 5)                       # re-seed: this section does not depend on earlier cells
N_IND, J_SNP = 600, 10
G, _, _ = simulate_genotypes(rng, N_IND, J_SNP, maf_range=(0.2, 0.5))
truth = ((G[:, 0] >= 1) & (G[:, 1] >= 1)) | (G[:, 2] == 2)     # the epistatic + recessive rule
flipped = rng.random(N_IND) < 0.15                              # 15 % label noise
y_gen = (truth ^ flipped).astype(int)                           # 1 = case, 0 = control
SNP_NAMES = [f"SNP{j + 1}" for j in range(J_SNP)]

perm = rng.permutation(N_IND)
i_tr, i_va, i_te = perm[:300], perm[300:450], perm[450:]
print("SIMULATED genotype study:", G.shape, "| fraction of cases:", y_gen.mean().round(3))
print("labels flipped:", flipped.sum(), f"→ best achievable test accuracy ≈ {1 - flipped[i_te].mean():.3f}"
      " (the true rule applied to the test set)")

In [ ]:
# ── 5.3 Accuracy vs tree size: the overfitting curve ──
growth = []
for max_depth in range(0, 9):
    t = id3(G[i_tr], y_gen[i_tr], list(range(J_SNP)), max_depth=max_depth)
    growth.append((max_depth, count_nodes(t), accuracy(y_gen[i_tr], predict_tree(t, G[i_tr])),
                   accuracy(y_gen[i_te], predict_tree(t, G[i_te]))))
growth = pd.DataFrame(growth, columns=["max_depth", "nodes", "train_acc", "test_acc"])
print(growth.round(3).to_string(index=False))
full_tree = id3(G[i_tr], y_gen[i_tr], list(range(J_SNP)))      # no depth limit = plain ID3

fig, ax = plt.subplots(figsize=(5.8, 3.4))
ax.plot(growth.nodes, growth.train_acc, "o-", color="tab:blue", label="on training data")
ax.plot(growth.nodes, growth.test_acc, "s--", color="tab:orange", label="on test data")
ax.axhline(1 - flipped[i_te].mean(), color="gray", lw=0.8, ls=":", label="true rule on test data")
ax.set_xlabel("size of tree (number of nodes)"); ax.set_ylabel("accuracy")
ax.set_title("SIMULATED epistasis study: bigger trees fit noise"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** As the tree grows, training accuracy climbs towards 100 % (it stops just short because a
few individuals share identical genotypes but different labels). Test accuracy rises at first — the first splits
find SNP3, SNP2 and SNP1 — peaks at depth 3 (34 nodes, test accuracy 0.873, as good as the true rule itself), and
then **falls** as the tree carves out tiny regions to accommodate flipped labels. The fully grown tree has 156 nodes,
fits 99.3 % of the training individuals, scores only 0.700 on the test set, and
predicts new individuals clearly worse than the depth-3 tree. That is the classic overfitting curve of decision-tree
learning (Mitchell's book draws it for a medical-diagnosis task), reproduced on genotypes.

⚠️ **The test set was used here only to draw the curve.** If we used it to *choose* the depth, its accuracy would no
longer be an honest estimate of performance on new patients. Choosing needs a separate **validation** set — next.

✏️ **Your turn.** Go back to 5.1 and give D15 the label *Yes* instead (`tennis15.loc[14, "PlayTennis"] = "Yes"` before
building `X15, y15`). How many nodes does the tree have now, and why does this "noisy" day no longer change anything?

---
## 6 · Avoiding overfitting: early stopping and reduced-error pruning

> 📍 **Where we are.** There are two strategies: **stop growing** when a split is not statistically significant
> (or the node is small, or a depth limit is reached), or **grow the full tree and then prune it**. Post-pruning is
> usually preferred because a split that looks useless on its own can enable very useful splits below it — the
> epistatic rule is exactly such a case: SNP1 alone is weakly informative, SNP1 *within* SNP2 carriers is decisive.

**How do we select the best tree** — by accuracy on the training data or on held-out data? Training accuracy
always prefers the biggest tree, so the choice must be made on data the tree was not grown on.
**Reduced-error pruning** does exactly that:

1. Split the data into a training set and a held-out set.
2. Grow a tree that classifies all training data correctly (as far as possible).
3. Repeat until further pruning is harmful: for every internal node, evaluate on the held-out set the effect of
   replacing that node *and everything below it* by a leaf predicting the node's majority label; greedily apply the
   replacement that most improves held-out accuracy (we also accept replacements that leave it unchanged, as
   Mitchell does — a smaller tree with equal accuracy is preferable).

⚠️ **Many textbooks call the held-out set the "test set".** Because pruning *chooses* the tree by its accuracy on that set,
the set becomes part of the training procedure. To report an honest accuracy you need a third, untouched set.
Here: train (300) → grow; **validation** (150) → prune; test (150) → report once.

In [ ]:
# ── 6.1 Reduced-error pruning ──
def internal_paths(node: dict, path=()) -> list:
    """Paths (tuples of branch values) to every internal node."""
    if "feature" not in node:
        return []
    out = [path]
    for v, child in node["children"].items():
        out += internal_paths(child, path + (v,))
    return out


def prune_at(tree: dict, path) -> dict:
    """Copy of the tree in which the node at `path` becomes a leaf (it keeps its majority label)."""
    new = copy.deepcopy(tree)
    node = new
    for v in path:
        node = node["children"][v]
    for key in ("feature", "gain", "children"):
        node.pop(key)
    return new


def reduced_error_prune(tree: dict, X_val, y_val):
    """Greedy reduced-error pruning. Returns (pruned tree, history of (nodes, validation accuracy, tree))."""
    history = [(count_nodes(tree), accuracy(y_val, predict_tree(tree, X_val)), tree)]
    while True:
        candidates = [(accuracy(y_val, predict_tree(prune_at(tree, p), X_val)), p) for p in internal_paths(tree)]
        if not candidates:
            break
        best_acc, best_path = max(candidates, key=lambda c: (c[0], -len(c[1])))   # prefer pruning near the root
        if best_acc < history[-1][1]:
            break                                            # every pruning step would hurt → stop
        tree = prune_at(tree, best_path)
        history.append((count_nodes(tree), best_acc, tree))
    return tree, history


t0 = time.time()
pruned_tree, prune_hist = reduced_error_prune(full_tree, G[i_va], y_gen[i_va])
print(f"pruning took {time.time() - t0:.1f} s and {len(prune_hist) - 1} steps")
for name, t in [("full tree", full_tree), ("pruned tree", pruned_tree)]:
    print(f"{name:12s}: {count_nodes(t):4d} nodes | train {accuracy(y_gen[i_tr], predict_tree(t, G[i_tr])):.3f}"
          f" | validation {accuracy(y_gen[i_va], predict_tree(t, G[i_va])):.3f}"
          f" | test {accuracy(y_gen[i_te], predict_tree(t, G[i_te])):.3f}")

In [ ]:
# ── 6.2 The effect of reduced-error pruning ──
sizes = [h[0] for h in prune_hist]
val_acc = [h[1] for h in prune_hist]
test_acc_pruning = [accuracy(y_gen[i_te], predict_tree(h[2], G[i_te])) for h in prune_hist]

fig, ax = plt.subplots(figsize=(6.2, 3.5))
ax.plot(growth.nodes, growth.train_acc, "o-", color="tab:blue", label="growing: training data")
ax.plot(growth.nodes, growth.test_acc, "s--", color="tab:orange", label="growing: test data")
ax.plot(sizes, test_acc_pruning, "^:", color="tab:green", label="pruning: test data")
ax.plot(sizes, val_acc, "v:", color="tab:purple", alpha=0.7, label="pruning: validation data")
ax.set_xlabel("size of tree (number of nodes)"); ax.set_ylabel("accuracy")
ax.set_title("reduced-error pruning walks back to a small tree (SIMULATED)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()
print_tree(pruned_tree, SNP_NAMES)

🔍 **What just happened?** Starting from the full tree (right end of the green and purple curves), each pruning step
removes a whole subtree; 11 steps shrink the tree from 156 to 33 nodes while validation accuracy rises from 0.693 to
0.853 and test accuracy from 0.700 to 0.873. Read the printed pruned tree: the root is SNP3 (the recessive locus:
genotype 2 → case, 44 individuals), and below it SNP2 and SNP1 — the epistatic pair; for SNP3 = 1 the subtree is
exactly the planted rule. Two small noise-fitted splits survive under SNP3 = 0: an SNP4 → SNP9 → SNP5 branch for
SNP1 = 0 (the planted rule calls all 37 of these training individuals controls; the tree calls 2 of them cases), and
an SNP4 split for SNP1 = 1, SNP2 = 2 that calls 1 of 8 training individuals a control. Apart from these few individuals, pruning has
recovered the rule we planted, from noisy labels, without being told which SNPs matter.

⚠️ **Honest caveats.** (i) The validation accuracy is optimistic — the tree was *selected* on it; the test accuracy is
the honest number. (ii) The planted rule itself scores 0.873 on this
test set (the flipped labels are unpredictable), so the pruned tree has reached the ceiling — it cannot do better.
(iii) A tree is an association model: a split on a SNP means the SNP *predicts* disease in this sample, not that it causes it (Module 06 discusses linkage and confounding).

✏️ **Your turn.** Set the label-noise rate to 0 (`flipped = rng.random(N_IND) < 0.0`) and re-run Sections 5.2–6.3.
How large is the fully grown tree now, and how much does pruning remove? Then try 30 % noise.

In [ ]:
# ── 6.3 🧪 Ground-truth check: does each tree reproduce the planted (noise-free) rule? ──
for name, t in [("full tree", full_tree), ("pruned tree", pruned_tree)]:
    agree_test = np.mean(predict_tree(t, G[i_te]) == truth[i_te])
    print(f"{name:12s}: agrees with the true rule on {agree_test:.1%} of the test individuals")
print("(the noisy test labels agree with the true rule on", f"{np.mean(y_gen[i_te] == truth[i_te]):.1%})")

🔍 **What just happened?** Because we *simulated* the study, we can score each tree against the truth itself rather
than against noisy labels. The pruned tree reproduces the planted rule for every one of the 150 test individuals
(100 %), whereas the fully grown tree contradicts it for 23 % of them — its extra nodes encode label noise, not
biology. With real data
this check is impossible, which is why held-out accuracy is the best we can usually do.

### 📘 Lecture notes — noise, overfitting and pruning

**One noisy example.** Add D15 = (Sunny, Hot, Normal, Strong) → *No* to PlayTennis. It falls into the
*Sunny → Normal → Yes* leaf with the wrong label. Do we want to be accurate on D15? *No* → keep the small tree;
*yes* → accept a bigger tree, which **overfits** if D15 is noise. Usually we cannot know ahead of time whether an
example is noise. (ID3 grows the tree from 8 to 11 nodes and even changes the test at the Sunny node to
Temperature.)

**Overfitting.** A model overfits when its error on the training data is low but its error on all data (the
population) is high. To detect it, split the available data into training and held-out data. The classic picture:
as the tree grows, training accuracy keeps rising while held-out accuracy rises, peaks and falls.

**Two remedies.**
1. *Early stopping:* stop growing when a split is not statistically significant (or a node is small, or a depth
   limit is reached). Risk: a weak split can enable strong splits below it (epistasis).
2. *Post-pruning:* grow the full tree, then prune. **Reduced-error pruning:** (1) split the data into training and
   held-out sets; (2) grow a tree that classifies all training data correctly; (3) until further pruning is
   harmful, evaluate on the held-out set the effect of replacing each internal node (and everything below it) by
   a majority-label leaf, and greedily remove the one that most improves held-out accuracy.

**Worked example (simulated epistasis study, 300/150/150 individuals, 15 % label noise).** The full tree has
156 nodes (train 0.993, test 0.700); depth 3 is best while growing (34 nodes, test 0.873); reduced-error
pruning on the validation set shrinks it to 33 nodes with test accuracy 0.873 — the ceiling set by the noise —
and recovers the planted rule "(SNP1 ≥ 1 AND SNP2 ≥ 1) OR SNP3 = 2".

**Common confusion.** The set used for pruning *chooses* the tree, so its accuracy is optimistic; it is a
**validation** set even when a textbook calls it the "test set". Report performance on a third, untouched set.

> 🎓 **Exam tip.** An exam may ask why the accuracy measured on the set used for pruning is optimistic, why you need a third set
    to report performance, and why post-pruning is usually preferred to early stopping (a split that looks useless
    alone can make the splits below it very useful).

---
## 7 · Continuous inputs: thresholds and rectangles

> 📍 **Where we are.** Expression levels, blood pressure and nuclear sizes are numbers, not categories. A tree
> discretises a continuous attribute with a **threshold**: the question becomes "Temperature > θ?", a binary attribute
> that information gain can score like any other.

Which thresholds should we try? The lecture's example records the temperature (°F) on six days and sorts them:

| Temperature | 40 | 48 | 60 | 72 | 80 | 90 |
|---|---|---|---|---|---|---|
| PlayTennis | No | No | Yes | Yes | Yes | No |

Candidate thresholds lie **halfway between neighbouring values where the label changes**: (48 + 60)/2 = 54 and
(80 + 90)/2 = 85. (A classic result of Fayyad & Irani (1992): the gain-maximising threshold always sits at such a
label boundary, so other midpoints never need to be checked. Any value between 48 and 60 — the lecture writes
"Temperature > 55?", which turns the row into F, F, T, T, T, T — makes the same split of these six days.)

In [ ]:
# ── 7.1 Gain of every candidate threshold on the six days ──
temp = np.array([40, 48, 60, 72, 80, 90])
play = np.array(["No", "No", "Yes", "Yes", "Yes", "No"])
for theta in (temp[:-1] + temp[1:]) / 2:                     # all midpoints between sorted values
    boundary = play[temp < theta][-1] != play[temp > theta][0]
    print(f"Temperature > {theta:5.1f}?  gain = {information_gain(temp > theta, play):.3f} bits"
          f"   {'← label changes here' if boundary else ''}")

🔍 **What just happened?** The threshold 54 has the highest gain (0.459 bits): "Temperature > 54?" separates the two
cold *No* days from the rest. The other boundary candidate, 85, gains 0.191 bits; the midpoint 76, inside a run of
*Yes* days, gains nothing. Note that the threshold 44 also reaches 0.191 — a midpoint between two equal labels can tie
a boundary candidate, but (Fayyad–Irani) it can never *beat* the best one.

### 7.2 A tree learner for numeric features

For numeric features we use **binary** splits $x_j \le \theta$ vs $x_j > \theta$ and search all features and all
midpoints. Sorting each feature once and using running class counts makes the search fast: after sorting, the
examples to the left of cut position $i$ are simply the first $i$ examples. The same function will be the building
block of our random forest, so it already accepts an option `m_features` (try only $m$ randomly chosen features at
each node — Section 9).

In [ ]:
# ── 7.2 Best threshold for one feature, and a numeric tree learner ──
def entropy_from_counts(counts) -> np.ndarray:
    """Entropy (bits) of each row of a (k, n_classes) array of class counts."""
    counts = np.atleast_2d(counts).astype(float)
    p = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        return -np.where(p > 0, p * np.log2(p), 0.0).sum(axis=1)


def best_threshold(x, y, n_classes: int):
    """(gain, threshold) of the best binary split x <= θ for integer labels y; (0, None) if x is constant."""
    order = np.argsort(x, kind="stable")
    xs, onehot = x[order], np.eye(n_classes)[y[order]]
    left = np.cumsum(onehot, axis=0)[:-1]                    # class counts of the first 1, 2, …, N−1 examples
    total = left[-1] + onehot[-1]
    n_left = np.arange(1, len(x))
    h_split = (n_left * entropy_from_counts(left) + (len(x) - n_left) * entropy_from_counts(total - left)) / len(x)
    gain = entropy_from_counts(total)[0] - h_split
    gain[xs[1:] == xs[:-1]] = -np.inf                         # cannot cut between equal values
    i = int(np.argmax(gain))
    return (0.0, None) if not np.isfinite(gain[i]) else (float(gain[i]), float((xs[i] + xs[i + 1]) / 2))


def grow_tree(X, y, n_classes: int, depth: int = 0, max_depth=None, min_samples_split: int = 2,
              m_features=None, rng=None) -> dict:
    """Binary decision tree on numeric features with the entropy criterion (ID3/C4.5-style).

    m_features: if given, only m randomly chosen features are tried at each node (random forests).
    """
    counts = np.bincount(y, minlength=n_classes)
    node = {"counts": counts, "prediction": int(np.argmax(counts))}
    if counts.max() == len(y) or len(y) < min_samples_split or (max_depth is not None and depth >= max_depth):
        return node
    features = range(X.shape[1]) if m_features is None else rng.choice(X.shape[1], m_features, replace=False)
    gain, theta, feat = max(((*best_threshold(X[:, f], y, n_classes), f) for f in features), key=lambda t: t[0])
    if theta is None or gain <= 0:
        return node
    left = X[:, feat] <= theta
    kw = dict(max_depth=max_depth, min_samples_split=min_samples_split, m_features=m_features, rng=rng)
    node.update(feature=int(feat), threshold=theta, gain=gain,
                left=grow_tree(X[left], y[left], n_classes, depth + 1, **kw),
                right=grow_tree(X[~left], y[~left], n_classes, depth + 1, **kw))
    return node

In [ ]:
# ── 7.3 Prediction, size and printing for numeric trees ──
def predict_numeric(node: dict, X) -> np.ndarray:
    """Predicted class of every row of X (examples are routed down the tree in vectorised batches)."""
    out = np.empty(len(X), dtype=int)

    def route(nd, idx):
        if "feature" not in nd:
            out[idx] = nd["prediction"]
            return
        go_left = X[idx, nd["feature"]] <= nd["threshold"]
        route(nd["left"], idx[go_left])
        route(nd["right"], idx[~go_left])

    route(node, np.arange(len(X)))
    return out


def n_nodes(node: dict) -> int:
    """Number of nodes of a numeric tree."""
    return 1 if "feature" not in node else 1 + n_nodes(node["left"]) + n_nodes(node["right"])


def print_numeric_tree(node: dict, names, classes, max_depth: int = 3, depth: int = 0, label: str = "") -> None:
    """Pretty-print the top `max_depth` levels of a numeric tree."""
    pad = "      " * depth + label
    if "feature" not in node or depth == max_depth:
        more = "   (deeper splits not shown)" if "feature" in node else ""
        print(f"{pad}→ {classes[node['prediction']]}   class counts {node['counts'].tolist()}{more}")
        return
    print(f"{pad}{names[node['feature']]} ≤ {node['threshold']:.2f} ?   (gain {node['gain']:.3f} bits)")
    print_numeric_tree(node["left"], names, classes, max_depth, depth + 1, "yes: ")
    print_numeric_tree(node["right"], names, classes, max_depth, depth + 1, "no:  ")

### 7.4 Rectangles on a flow-cytometry scatter plot

The lecture draws the boundaries of a tree on Fisher's iris flowers: two measurements per flower (sepal length and
width), with *Iris setosa* to be separated from the other two species; the first two cuts — "sepal length < 5.7",
then "sepal width > 2.8" — box in the setosa cloud. Each threshold test is a vertical or horizontal line, so a tree
partitions the plane into **axis-parallel rectangles**.

A biologist meets exactly this picture in **flow cytometry**. Every cell passing the laser gives a **forward-scatter**
signal (FSC, roughly cell size) and a **side-scatter** signal (SSC, granularity); on an FSC–SSC plot lymphocytes
(small, agranular), monocytes (larger) and granulocytes (granular) form three clouds. Gating them by hand means
drawing rectangles — a decision tree drawn by a human. We let the computer draw the gates on `simulate_blood_cells`:
150 SIMULATED cells, 50 of each type, and separate the **granulocytes** from the rest (like iris, one cloud is easy to
separate — the lymphocytes — and two overlap a little).

In [ ]:
# ── 7.4 A tree on FSC and SSC (granulocytes vs the rest, SIMULATED), checked against scikit-learn ──
blood = simulate_blood_cells()
print("SIMULATED flow cytometry:", blood["X"].shape[0], "cells × markers", list(blood["X"].columns))
print("cell types:", dict(zip(blood["class_names"], np.bincount(blood["y"]).tolist())))
print(blood["X"].groupby(blood["class_names"][blood["y"]]).mean().round(2).to_string())

X_fc = blood["X"][["FSC", "SSC"]].to_numpy()                 # forward scatter (size), side scatter (granularity)
y_fc = (blood["y"] == 2).astype(int)                         # 1 = granulocyte, 0 = lymphocyte or monocyte
FC_FEATS, FC_CLASSES = ["FSC", "SSC"], ["other", "granulocyte"]

tree_fc = grow_tree(X_fc, y_fc, 2)
print()
print_numeric_tree(tree_fc, FC_FEATS, FC_CLASSES, max_depth=2)
sk_tree = DecisionTreeClassifier(criterion="entropy", random_state=SEED).fit(X_fc, y_fc)
print(f"\nour tree: {n_nodes(tree_fc)} nodes | scikit-learn (criterion='entropy'): {sk_tree.tree_.node_count} nodes;"
      f" first thresholds {np.round(sk_tree.tree_.threshold[:2], 2)} on features {sk_tree.tree_.feature[:2]}")
print("training accuracy:", accuracy(y_fc, predict_numeric(tree_fc, X_fc)))

tree3c = grow_tree(X_fc, blood["y"], 3, max_depth=2)          # all three cell types, depth 2
print("\nthree cell types, depth 2:")
print_numeric_tree(tree3c, FC_FEATS, list(blood["class_names"]), max_depth=2)

In [ ]:
# ── 7.5 Axis-parallel rectangles vs a straight line ──
logreg_fc = LogisticRegression(C=1.0, max_iter=5000).fit(X_fc, y_fc)
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9))
plot_decision_regions(lambda Z: predict_numeric(tree_fc, Z), X_fc, y_fc, ax=axes[0], pad=0.3, class_names=FC_CLASSES,
                      title=f"SIMULATED cells — decision tree ({n_nodes(tree_fc)} nodes):\nrectangles, "
                            f"training accuracy {accuracy(y_fc, predict_numeric(tree_fc, X_fc)):.3f}")
plot_decision_regions(logreg_fc.predict, X_fc, y_fc, ax=axes[1], pad=0.3, class_names=FC_CLASSES,
                      title=f"logistic regression: one straight line,\ntraining accuracy {logreg_fc.score(X_fc, y_fc):.3f}")
plot_decision_regions(lambda Z: predict_numeric(tree3c, Z), X_fc, blood["y"], ax=axes[2], pad=0.3,
                      class_names=list(blood["class_names"]), title="three cell types, depth-2 tree:\nautomated gating")
for ax in axes:
    ax.set_xlabel("forward scatter FSC (size)"); ax.set_ylabel("side scatter SSC (granularity)")
axes[0].legend(loc="upper left", fontsize=7); axes[2].legend(loc="upper left", fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Our tree agrees with scikit-learn's entropy tree (same number of nodes, same first two
thresholds), a useful check of the from-scratch code. Its first questions are *SSC ≤ 2.98* and then *SSC ≤ 2.39*: the
tree gates on **granularity**, the property granulocytes are named after, and uses size (FSC) only inside the zone
where monocytes and granulocytes overlap. With 25 nodes it classifies every training cell correctly — including one
granulocyte with unusually low side scatter (FSC ≈ 4.6, SSC ≈ 2.4), which gets its own thin horizontal strip across
the whole plot (the thick line): a first glimpse of overfitting. Logistic regression draws one oblique line with far
fewer "decisions" and accepts a training accuracy of 0.947. The right panel is the three-class version: the first cut
(FSC ≤ 3.60) isolates all 50 lymphocytes, the second (SSC ≤ 3.20) separates monocytes from granulocytes with 7 of
the 150 cells on the wrong side — much as a cytometrist would gate them.

### 7.6 Where trees struggle

Decision trees do not typically improve over linear classifiers when there are lots of features, and they sometimes
fail badly on problems that linear classifiers solve well. If the true boundary is **oblique** — say a cell type is
defined by the *sum* of two marker genes — a tree must approximate the diagonal with many small steps.

In [ ]:
# ── 7.6 An oblique boundary: staircase vs line (SIMULATED) ──
rng = np.random.default_rng(SEED + 7)
def two_marker_cells(n: int):
    """SIMULATED: standardised levels of two marker genes; cell type 1 if their sum is positive."""
    Xc = rng.normal(0, 1, (n, 2))
    return Xc, (Xc[:, 0] + Xc[:, 1] > 0).astype(int)


X_obl, y_obl = two_marker_cells(100)
X_obl_test, y_obl_test = two_marker_cells(2000)
tree_obl = grow_tree(X_obl, y_obl, 2)
logreg_obl = LogisticRegression(C=1.0, max_iter=5000).fit(X_obl, y_obl)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, f, name in [(axes[0], lambda Z: predict_numeric(tree_obl, Z), f"tree ({n_nodes(tree_obl)} nodes)"),
                    (axes[1], logreg_obl.predict, "logistic regression")]:
    acc = accuracy(y_obl_test, f(X_obl_test))
    plot_decision_regions(f, X_obl, y_obl, ax=ax, pad=0.3, title=f"{name}: test accuracy {acc:.3f}")
    ax.plot([-3, 3], [3, -3], "k:", lw=1)                     # the true boundary x1 + x2 = 0
    ax.set_xlabel("marker gene 1 (z-score)"); ax.set_ylabel("marker gene 2 (z-score)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With 100 training cells the tree approximates the diagonal (dotted) with a staircase of
13 nodes and misclassifies 8 % of 2,000 new cells; logistic regression recovers an almost exact copy of the
true line and scores 0.975. Trees shine when the truth is made of thresholds and interactions; linear models shine when
the truth is an additive score. Neither is "better" in general — which is why model choice must be checked on held-out
data (Module 23).

✏️ **Your turn.** With two real-valued inputs $x_1, x_2$, which decision boundaries can a tree draw if it may use
each attribute **at most once on any path**? Grow `grow_tree(X_obl, y_obl, 2, max_depth=2)`, plot it, and describe the
family of shapes you can get. (Hint: one cut on $x_1$, then on each side one cut on $x_2$.)

### 📘 Lecture notes — continuous inputs, rectangles and rules

**Thresholds.** A continuous attribute is discretised by a test "$x > \theta$?". Candidate thresholds are the
midpoints between neighbouring sorted values **where the label changes** (Fayyad and Irani: the gain-maximising
threshold is always one of them).

**Worked example.** Temperatures 40, 48, 60, 72, 80, 90 with labels No, No, Yes, Yes, Yes, No. Candidates
(48 + 60)/2 = 54 and (80 + 90)/2 = 85; "Temperature > 54?" (or > 55 — any cut between 48 and 60) gives
F, F, T, T, T, T and gain 0.459 bits, "> 85?" only 0.191 bits.

**Geometry.** Each test on one numeric attribute is a vertical or horizontal line, so a tree partitions the
input space into **axis-parallel rectangles** (boxes). The lecture's iris example: "sepal length < 5.7" then
"sepal width > 2.8" box in *Iris setosa*; here, cuts on side scatter gate granulocytes on a flow-cytometry plot. A
linear classifier draws one oblique line instead. Trees do not typically beat linear classifiers when there are
many features, and they can fail badly on oblique problems, which they approximate with a staircase.

**Each attribute at most once per path** (two real inputs): one cut on $x_1$, then at most one cut on $x_2$ on
each side — boundaries made of at most three axis-parallel segments (up to four rectangles).

**Trees = IF-THEN rules.** One rule per leaf: the tests along the path joined by AND. Ordered rule lists can be
simplified, e.g. "if Outlook = sunny and Humidity = high then don't play; else if Outlook = rain and windy then
don't play; else play".

**Common confusion.** A threshold between two *equal-label* neighbours can tie the best boundary candidate but
never beat it; and a tree's rectangles fit the training points perfectly long before they describe the
population (Sections 5 and 8).

> 🎓 **Exam tip.** Be able to list the candidate thresholds for a sorted numeric attribute (midpoints at label changes), compute
    the gain of one of them, and sketch the rectangular regions of a small tree in a 2-D plot.

---
## 8 · Pros, cons and the variance problem

> 📍 **Where we are.** Time to take stock. **Pros** of decision trees: the tree structure is easy to interpret; they
> handle mixed discrete and continuous inputs; they perform automatic variable selection; they scale well to large
> data. **Cons:** prediction accuracy is often lower than that of other classifiers; **high variance** — small changes
> in the data cause large changes in the tree structure; and an error at the top of the tree affects the rest of the
> tree-growing process. The lecture illustrates the variance problem by adding a single data point and redrawing the
> boundaries of a tree and of logistic regression.

A model has **high variance** if refitting it on a slightly different sample of the same population gives a very
different model; a **low-variance** model is robust to a small change in the data (Module 23 makes this precise with
the bias–variance decomposition). We test it twice: by adding one granulocyte that the current tree calls "other"
(at FSC 4.0, SSC 3.3), and by refitting on bootstrap resamples.

In [ ]:
# ── 8.1 Add one cell ──
extra = np.array([[4.0, 3.3]])                                # a granulocyte in 'other' territory (our choice)
print("the original tree calls the added cell:", FC_CLASSES[predict_numeric(tree_fc, extra)[0]])
X_plus, y_plus = np.vstack([X_fc, extra]), np.r_[y_fc, 1]
tree_plus = grow_tree(X_plus, y_plus, 2)
logreg_plus = LogisticRegression(C=1.0, max_iter=5000).fit(X_plus, y_plus)

lo, hi = X_fc.min(axis=0) - 0.3, X_fc.max(axis=0) + 0.3
gx, gy = np.meshgrid(np.linspace(lo[0], hi[0], 300), np.linspace(lo[1], hi[1], 300))
grid = np.c_[gx.ravel(), gy.ravel()]
changed_tree = np.mean(predict_numeric(tree_fc, grid) != predict_numeric(tree_plus, grid))
changed_lr = np.mean(logreg_fc.predict(grid) != logreg_plus.predict(grid))
print(f"tree: {n_nodes(tree_fc)} → {n_nodes(tree_plus)} nodes; predictions change on {changed_tree:.1%} of the plane")
print(f"logistic regression: predictions change on {changed_lr:.1%} of the plane")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, before, after, title in [
        (axes[0], lambda Z: predict_numeric(tree_fc, Z), lambda Z: predict_numeric(tree_plus, Z), "decision tree"),
        (axes[1], logreg_fc.predict, logreg_plus.predict, "logistic regression")]:
    ax.scatter(*X_fc[y_fc == 1].T, s=12, color=CLASS_COLORS[1], label="granulocyte")
    ax.scatter(*X_fc[y_fc == 0].T, s=12, color=CLASS_COLORS[0], label="other")
    changed = (before(grid) != after(grid)).reshape(gx.shape)
    ax.contourf(gx, gy, changed.astype(float), levels=[0.5, 1.5], colors=["#fdae6b"], alpha=0.5)
    ax.contour(gx, gy, before(grid).reshape(gx.shape), levels=[0.5], colors="0.3", linewidths=3, alpha=0.5)
    ax.contour(gx, gy, after(grid).reshape(gx.shape), levels=[0.5], colors="tab:red", linewidths=1.0)
    ax.scatter(*extra.T, s=120, facecolor="none", edgecolor="tab:red", lw=2, label="added cell")
    ax.set_title(f"{title}: grey = before, red = after,\norange = predictions changed ({changed.mean():.1%})")
    ax.set_xlabel("forward scatter FSC")
    ax.set_ylabel("side scatter SSC")
axes[0].legend(fontsize=7, loc="upper left"); plt.tight_layout(); plt.show()

In [ ]:
# ── 8.2 Refit on 20 bootstrap resamples and measure the instability ──
rng = np.random.default_rng(SEED + 8)
votes_tree, votes_lr = [], []
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for b in range(20):
    boot = rng.integers(0, len(y_fc), len(y_fc))              # N cells drawn with replacement
    pt = predict_numeric(grow_tree(X_fc[boot], y_fc[boot], 2), grid)
    pl = LogisticRegression(C=1.0, max_iter=5000).fit(X_fc[boot], y_fc[boot]).predict(grid)
    votes_tree.append(pt); votes_lr.append(pl)
    if b < 8:
        axes[0].contour(gx, gy, pt.reshape(gx.shape), levels=[0.5], colors="tab:green", linewidths=0.8, alpha=0.7)
        axes[1].contour(gx, gy, pl.reshape(gx.shape), levels=[0.5], colors="tab:green", linewidths=0.8, alpha=0.7)
for ax, v, name in [(axes[0], np.array(votes_tree), "trees"), (axes[1], np.array(votes_lr), "logistic regression")]:
    unstable = np.mean((v.mean(0) > 0) & (v.mean(0) < 1))
    ax.scatter(*X_fc.T, s=6, c=[CLASS_COLORS[k] for k in y_fc])
    ax.set_title(f"{name} on 8 bootstrap samples\nplane where 20 refits disagree: {unstable:.0%}")
    ax.set_xlabel("forward scatter FSC"); ax.set_ylabel("side scatter SSC")
plt.tight_layout(); plt.show()

🔍 **What just happened?** One extra granulocyte makes the tree grow from 25 to 27 nodes and change its prediction
over 4.6 % of the plotted plane (shaded in orange): it re-cuts the overlap zone and adds a strip across the whole
width around SSC ≈ 3.3.
Logistic regression's line merely tilts, changing 1.3 % — a single cell, and the tree reacts about three and a half
times as strongly. The bootstrap experiment is more convincing than any single example: across 20 resamples of the
*same* 150 cells, the tree boundaries (green) wander all over the overlap zone, and the 20 trees disagree on 36 % of
the plotted plane; the logistic-regression lines stay in a narrow band and disagree on 10 %. Logistic regression is a
**low-variance** model, a fully grown tree a **high-variance** model.

Why? Each split is chosen among many thresholds that are almost equally good; a small change in the data flips the
winner, and since every later split depends on the earlier ones, the change cascades down the tree. The good news:
**variance can be averaged away.** That is the idea behind random forests.

✏️ **Your turn.** In 8.2 grow depth-2 trees instead (`grow_tree(X_fc[boot], y_fc[boot], 2, max_depth=2)`). Does
the fraction of the plane where the refits disagree go down? What does that tell you about tree size and variance?

### 📘 Lecture notes — pros, cons and the variance of trees

| Pros | Cons |
|---|---|
| the tree structure is easy to interpret | prediction accuracy is often lower than that of other classifiers |
| handles mixed discrete and continuous inputs | **high variance**: small changes in the data change the tree structure a lot |
| automatic variable selection (unused attributes drop out) | an error at the top of the tree affects everything grown below it |
| scales well to large data | |

**Low vs high variance.** A *low-variance* model (logistic regression) changes little when the data change a
little: add one point and its line merely tilts. A *high-variance* model (a fully grown tree) changes a lot: the
same point can rearrange the splits and the rectangles.

**Worked example (SIMULATED blood cells, granulocytes vs rest).** Adding one granulocyte grows the tree from 25
to 27 nodes and changes its predictions on 4.6 % of the plane (logistic regression: 1.3 %); across 20 bootstrap
refits the trees disagree on 36 % of the plane, the logistic-regression lines on 10 %.

**Why trees are unstable.** Many thresholds are almost equally good; a small change flips the winner, and every
split below depends on it. Trees have low bias (they can fit almost anything) but high variance — the opposite of
logistic regression.

**Common confusion.** High variance is about how much the *fitted model* changes between training samples, not
about noisy predictions for one fixed model. It is measured by refitting (bootstrap, cross-validation), not by
looking at one tree.

> 🎓 **Exam tip.** Know the definitions: high variance = the fitted model changes a lot when the training sample changes. Trees:
    low bias, high variance; logistic regression: the opposite. Be ready to argue why averaging many trees helps.

---
## 9 · Random forests: averaging many decorrelated trees

> 📍 **Where we are.** Section 8 showed that trees are unstable — a high-variance classifier. The remedy: build a
> **forest** of many decision trees and classify a test sample by the **majority vote** of all trees in the forest
> (Breiman 2001; *The Elements of Statistical Learning*, Algorithm 15.1). Here $N$ is the number of training samples
> and $M$ the number of input variables (ESL writes $p$; in the course's notation $M$ = J features).

For $b = 1, \dots, B$:

1. **Bootstrap.** Draw a sample $Z^*$ of $N$ training examples *at random with replacement* from the $N$ available.
   This sample is the training set for tree $T_b$.
2. **Grow a random-forest tree** $T_b$ on this sample by repeating, for each terminal node, until the minimum node
   size $n_{\min}$ is reached: (i) select $m \ll M$ variables at random from the $M$; (ii) pick the best
   variable/split point among those $m$; (iii) split the node into two daughter nodes. The value of $m$ is held
   constant while the forest grows. With $n_{\min} = 1$ each tree is grown **to the largest extent possible — no
   pruning**.

Output the ensemble of trees $\{T_b\}_{b=1}^{B}$; to classify a new example, let all $B$ trees vote.

### 9.1 Why averaging helps — and why trees must be decorrelated

🧮 **Math refresher — variance of an average.** The variance of a random quantity $T$ measures how much it fluctuates:
$\operatorname{Var}(T) = E[(T - E[T])^2]$. For two quantities, the **covariance** $\operatorname{Cov}(T_b, T_c)$
measures whether they fluctuate together, and the **correlation** $\rho = \operatorname{Cov}/\sigma^2$ (for equal
variances $\sigma^2$) puts it on a −1 … 1 scale. Two rules: $\operatorname{Var}(aT) = a^2\operatorname{Var}(T)$ and
$\operatorname{Var}(\sum_b T_b) = \sum_b \sum_c \operatorname{Cov}(T_b, T_c)$.

Think of $T_b$ as tree $b$'s prediction at some fixed input (a probability, say). The trees are identically
distributed, each with variance $\sigma^2$ and pairwise correlation $\rho$ (they are built from overlapping
bootstrap samples of the same data). For the forest's average:

$$\operatorname{Var}\Big(\frac1B\sum_{b=1}^{B} T_b\Big)
 = \frac{1}{B^2}\Big[\underbrace{B\,\sigma^2}_{b=c} + \underbrace{B(B-1)\,\rho\sigma^2}_{b\neq c}\Big]
 = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2 .$$

*In words:* the second term shrinks as you add trees, but the first does not — **correlation between trees sets a
floor** that no number of trees can break. Bagging (bootstrap aggregating) alone already averages; the random choice
of $m$ variables at each node makes the trees *less alike* (lower $\rho$), which lowers the floor. Let us check the
formula by simulation before trusting it.

In [ ]:
# ── 9.1 Check Var(average) = ρσ² + (1 − ρ)σ²/B by simulation ──
rng = np.random.default_rng(SEED + 9)
Bs = np.array([1, 2, 5, 10, 20, 50, 100])
fig, ax = plt.subplots(figsize=(5.6, 3.3))
for rho, col in [(0.0, "tab:blue"), (0.3, "tab:orange"), (0.7, "tab:red")]:
    shared = rng.normal(size=(4000, 1))                        # the part all 'trees' share
    own = rng.normal(size=(4000, Bs.max()))                    # each tree's own fluctuation
    T = np.sqrt(rho) * shared + np.sqrt(1 - rho) * own         # variance 1, pairwise correlation ρ
    simulated = [T[:, :B].mean(axis=1).var() for B in Bs]
    ax.plot(Bs, simulated, "o", color=col, label=f"simulated, ρ = {rho}")
    ax.plot(Bs, rho + (1 - rho) / Bs, "-", color=col, lw=1)
    print(f"ρ = {rho}: B = 100 → simulated {simulated[-1]:.3f}, formula {rho + (1 - rho) / 100:.3f}")
ax.set_xscale("log"); ax.set_xlabel("number of averaged trees B"); ax.set_ylabel("variance of the average (σ² = 1)")
ax.set_title("averaging helps until the correlation floor ρσ²"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Simulation (dots) and formula (lines) agree. With uncorrelated trees the variance falls
like $1/B$ towards zero; with $\rho = 0.7$ it never drops below 0.7 however many trees you add. Random forests attack
$\rho$ — that is the whole point of the random feature subsets.

### 9.2 Our own forest on simulated tumour data

We now grow forests from our `grow_tree` on tumour data shaped like the **Wisconsin diagnostic breast-cancer** study
(Wolberg, Street and Mangasarian 1995): fine-needle aspirates of breast masses, each digitised image summarised by 30
features of its cell nuclei — the mean, the standard error and the largest ("worst") value of radius, texture,
perimeter, area, smoothness, compactness, concavity, concave points, symmetry and fractal dimension. The toolkit's
`simulate_tumor_features` generates 569 SIMULATED tumours (212 malignant, 357 benign) with the same 30 features and
the same strong correlations (perimeter and area follow from radius; compactness, concavity and concave points share
an irregularity factor). Label 1 = malignant. We hold out 30 % as a test set and use $m = 5 \approx \sqrt{30}$, the
usual default for classification.

In [ ]:
# ── 9.2 A random forest from scratch ──
def random_forest(X, y, n_trees: int, m: int, rng, max_depth=None) -> list:
    """Grow n_trees randomised trees on bootstrap samples. Returns a list of (tree, bootstrap indices)."""
    forest = []
    for _ in range(n_trees):
        boot = rng.integers(0, len(y), len(y))                 # bootstrap: N draws with replacement
        forest.append((grow_tree(X[boot], y[boot], 2, max_depth=max_depth, m_features=m, rng=rng), boot))
    return forest


def tree_votes(forest: list, X) -> np.ndarray:
    """(n_trees, n_samples) array of 0/1 predictions."""
    return np.array([predict_numeric(t, X) for t, _ in forest])


tumours = simulate_tumor_features()
print(tumours["source"])
X_bc, y_bc = tumours["X"].to_numpy(), tumours["y"]           # 1 = malignant, 0 = benign
BC_FEATS = list(tumours["X"].columns)
rng = np.random.default_rng(SEED + 92)
tr_bc, te_bc = train_test_indices(len(y_bc), 0.3, rng, stratify=y_bc)
print(f"{X_bc.shape[0]} tumours × {X_bc.shape[1]} features; malignant fraction {y_bc.mean():.2f};"
      f" first features: {BC_FEATS[:4]}")

t0 = time.time()
forest = random_forest(X_bc[tr_bc], y_bc[tr_bc], n_trees=100, m=5, rng=rng)
votes = tree_votes(forest, X_bc[te_bc])
print(f"grew 100 trees in {time.time() - t0:.1f} s; mean size {np.mean([n_nodes(t) for t, _ in forest]):.0f} nodes")
single = grow_tree(X_bc[tr_bc], y_bc[tr_bc], 2)
print(f"one full tree (all features): test accuracy {accuracy(y_bc[te_bc], predict_numeric(single, X_bc[te_bc])):.3f}")
print(f"individual forest trees: mean test accuracy {np.mean(votes == y_bc[te_bc]):.3f}")
print(f"forest majority vote   : test accuracy {accuracy(y_bc[te_bc], (votes.mean(0) > 0.5).astype(int)):.3f}")

In [ ]:
# ── 9.3 Accuracy vs number of trees, and the out-of-bag estimate ──
X_tr, y_tr = X_bc[tr_bc], y_bc[tr_bc]
B_grid = np.arange(1, len(forest) + 1)
test_acc_B, test_brier_B, oob_acc_B = [], [], []
oob_votes = np.zeros((len(y_tr), 2))                           # OOB votes for (benign, malignant)
for B, (t, boot) in zip(B_grid, forest):
    oob = np.setdiff1d(np.arange(len(y_tr)), boot)            # training tumours this tree never saw
    oob_votes[oob, predict_numeric(t, X_tr[oob])] += 1
    seen = oob_votes.sum(axis=1) > 0
    oob_acc_B.append(accuracy(y_tr[seen], oob_votes[seen].argmax(axis=1)))
    frac = votes[:B].mean(axis=0)                             # fraction of the first B trees voting 'malignant'
    test_acc_B.append(accuracy(y_bc[te_bc], (frac > 0.5).astype(int)))
    test_brier_B.append(np.mean((frac - y_bc[te_bc]) ** 2))   # squared distance of the vote fraction from the truth

oob_frac = np.mean([1 - len(np.unique(b)) / len(y_tr) for _, b in forest])
print(f"average fraction of training tumours left out of a bootstrap sample: {oob_frac:.3f} (theory 1 − 1/e ≈ 0.368)")
print(f"after 100 trees: OOB accuracy {oob_acc_B[-1]:.3f} ({len(y_tr)} training tumours) | "
      f"test accuracy {test_acc_B[-1]:.3f} ({len(te_bc)} test tumours; one tumour = {1 / len(te_bc):.1%})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].plot(B_grid, test_acc_B, color="tab:green", label=f"test accuracy ({len(te_bc)} tumours)")
axes[0].plot(B_grid, oob_acc_B, color="tab:purple", label=f"OOB accuracy ({len(y_tr)} tumours)")
axes[0].axhline(np.mean(votes == y_bc[te_bc]), color="tab:orange", ls="--", label="average single tree (test)")
axes[0].set_ylabel("accuracy"); axes[0].set_title("majority vote of the first B trees")
axes[1].plot(B_grid, test_brier_B, color="tab:red")
axes[1].set_ylabel("mean squared error of vote fraction"); axes[1].set_title("Brier score on the test tumours")
for ax in axes:
    ax.set_xlabel("number of trees B")
axes[0].legend(fontsize=7); plt.suptitle("SIMULATED breast tumours: our random forest", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In 9.2 each randomised tree saw only a bootstrap sample and 5 of the 30 features at each
split; on average such a tree (0.925) was a little less accurate on the test tumours than one ordinary tree grown on
all the data with all features (0.947) — and yet the **majority vote** of 100 of them was clearly better than either
(0.965). The panels show how the vote improves as trees are added. **Accuracy** is a jumpy measure on 171 test
tumours: each tumour is worth 0.6 percentage points, so the green curve moves in steps of one or two tumours (it
reaches about 0.955 within ten trees and then wobbles between 0.947 and 0.965). The **OOB accuracy**, computed on
398 training tumours, each predicted only by trees that did not see it, rises for the first couple of dozen trees and
then levels off around 0.93. The **Brier score** — the mean squared distance between the fraction of trees voting
"malignant" and the true label — falls steeply from above 0.07 and flattens near 0.028 after about 20 trees: the variance
reduction of Section 9.1 made visible. Adding trees never makes the forest overfit in the sense of Section 5; it only
shrinks the $(1-\rho)\sigma^2/B$ term, with diminishing returns.

The **out-of-bag** idea is a free validation set: each bootstrap sample leaves out about 37 % of the training
examples ($(1 - 1/N)^N \to 1/e$; here 0.366 on average), so each training tumour is predicted by roughly a third of
the trees. Here the OOB estimate (0.935) is lower than the test accuracy (0.965) — both are estimates from a few
hundred tumours, and a gap of three points is about five test tumours, within their sampling noise. OOB is handy for
choosing $m$ or tree depth without touching the test set.

### 9.4 Two knobs: m and tree depth

In [ ]:
# ── 9.4a The number of candidate features per split, m ──
rows = []
for m in [1, 5, 30]:                                           # m = 30 = M is plain bagging
    f_m = random_forest(X_tr, y_tr, n_trees=60, m=m, rng=np.random.default_rng(SEED + m))
    v_m = tree_votes(f_m, X_bc[te_bc])
    signed = np.where(v_m == y_bc[te_bc], 1.0, -1.0)             # +1 if a tree is right on a tumour, −1 if wrong
    corr = np.corrcoef(signed)[np.triu_indices(len(f_m), 1)]
    oob = np.zeros((len(y_tr), 2))
    for t, boot in f_m:
        idx = np.setdiff1d(np.arange(len(y_tr)), boot)
        oob[idx, predict_numeric(t, X_tr[idx])] += 1
    rows.append(dict(m=m, single_tree_acc=np.mean(v_m == y_bc[te_bc]), tree_correlation=np.nanmean(corr),
                     forest_OOB_acc=accuracy(y_tr, oob.argmax(1)),
                     forest_test_acc=accuracy(y_bc[te_bc], (v_m.mean(0) > 0.5).astype(int))))
print(pd.DataFrame(rows).round(3).to_string(index=False))

🔍 **What just happened?** The table shows the trade-off behind $m$. With $m = 1$ each split uses a random feature:
single trees are the weakest (0.907) but also the least alike (correlation 0.22). With $m = 30$ — plain bagging —
single trees are stronger (0.935) but more alike (0.39); $m = 5$ sits in between (0.926, 0.29). ("tree_correlation"
is the average correlation between two trees' right/wrong patterns over the test tumours — the quantity Breiman
(2001) relates to forest error together with tree *strength*.) The three forests end up within a few tumours of each
other: $m = 1$ and bagging tie on OOB accuracy (0.955), and bagging even has the best test accuracy (0.977, i.e. 4
errors against 6 for $m = 1$ and 8 for $m = 5$). On these simulated tumours many features carry nearly the same
information (radius, perimeter, area and their "worst" versions are almost copies of each other), so even bagged
trees are strong and decorrelation buys little — do not over-read one split. **$m$ is a tuning parameter**: choose it
with the OOB estimate or cross-validation, not by peeking at the test set.

The lecture shows the same trade-off in a simulation from *The Elements of Statistical Learning* (the "nested
spheres" example, Figure 15.3: forests of 500 trees, 50 simulated data sets), comparing "RF-1" with "RF-3". In that
book's notation the number is **$m$**, the number of candidate variables per split — RF-1 is a forest with $m = 1$,
RF-3 one with $m = 3$ — and the forest with $m = 1$ had a slightly lower test error. (Careful when revising: the
labels are sometimes read as tree depths, but "RF-$k$" means $m = k$.) So a small $m$ *can* win; on our tumours it
did not clearly — which is exactly why $m$ is tuned rather than fixed.

**Tree depth** is a different knob, and here it is our own question: does a forest need full-depth trees, or would
shallow trees do?

In [ ]:
# ── 9.4b Shallow vs full-depth trees inside a forest, 5 random splits ──
errors = {"depth 1": [], "depth 3": [], "full depth": []}
for s in range(5):
    r = np.random.default_rng(SEED + 100 + s)
    tr_s, te_s = train_test_indices(len(y_bc), 0.3, r, stratify=y_bc)
    for name, depth in [("depth 1", 1), ("depth 3", 3), ("full depth", None)]:
        v = tree_votes(random_forest(X_bc[tr_s], y_bc[tr_s], n_trees=60, m=5, rng=r, max_depth=depth), X_bc[te_s])
        errors[name].append(1 - accuracy(y_bc[te_s], (v.mean(0) > 0.5).astype(int)))
fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.boxplot(list(errors.values()))                              # labels set below: works on every Matplotlib ≥ 3.7
ax.set_xticks(range(1, len(errors) + 1), list(errors.keys()))
ax.set_ylabel("test misclassification error"); ax.set_title("forests of 60 trees, 5 splits (SIMULATED tumours)")
plt.tight_layout(); plt.show()
print("mean test error:", {k: round(float(np.mean(v)), 3) for k, v in errors.items()})

🔍 **What just happened?** On these tumours the three kinds of forest are close: forests of stumps (depth 1) err
slightly more often on average (6.4 %) than forests of depth-3 (5.6 %) or full-depth trees (5.3 %), and the boxes
overlap — five splits with 171 test tumours each cannot separate them. Very shallow trees are a little too simple
here, while depth 3 is already about as good as full depth; how deep the trees must be depends on how many features
have to be combined to describe the class boundary. **Tree depth is a model-selection knob** — in 02-620's model-selection lecture, "tree size" is the complexity
parameter of a decision tree (Module 23) — to be tuned on held-out data.

### 9.5 Check against scikit-learn

In [ ]:
# ── 9.5 Same recipe in scikit-learn (fast C implementation) ──
t0 = time.time()
rf_sk = RandomForestClassifier(n_estimators=100, criterion="entropy", max_features=5, oob_score=True,
                               random_state=SEED, n_jobs=1).fit(X_tr, y_tr)
print(f"scikit-learn forest: test accuracy {rf_sk.score(X_bc[te_bc], y_bc[te_bc]):.3f}, OOB {rf_sk.oob_score_:.3f}"
      f"  ({time.time() - t0:.1f} s)")
print(f"our forest         : test accuracy {test_acc_B[-1]:.3f}, OOB {oob_acc_B[-1]:.3f}")

🔍 **What just happened?** scikit-learn's `RandomForestClassifier` with the same settings (100 trees, entropy,
5 features per split, bootstrap) gives test and OOB accuracies within about one percentage point of our forest — the
remaining differences come from random seeds and tie-breaking. From now on we can use the fast library version.

✏️ **Your turn.** In 9.2 set `m=30` and `n_trees=1`: you get one bagged tree. Then set `n_trees=100`. How much of
the forest's advantage comes from bagging alone? (Compare with the table in 9.4a.)

### 📘 Lecture notes — random forests

**Idea.** A decision tree is a high-variance classifier (depending on the data, the tree can vary a lot), whereas
a low-variance classifier is robust to small changes in the data. A **random forest** builds a forest of many
decision trees and classifies a test sample by the **majority vote** of all trees.

**Training algorithm** ($N$ training cases, $M$ input variables, $B$ trees). For $b = 1, \dots, B$:
1. draw $N$ cases at random **with replacement** (a bootstrap sample) — the training set of tree $b$;
2. grow the tree: at each node select $m \ll M$ variables at random, split on the best of these $m$ ($m$ is held
   constant), and continue until the minimum node size is reached — each tree is grown to the largest extent
   possible, **no pruning**.

Output the ensemble $\{T_b\}$ and predict by majority vote. CHASM (Section 10) uses $B = 500$.

**Why it works.** For $B$ identically distributed trees with variance $\sigma^2$ and pairwise correlation $\rho$,

$$\operatorname{Var}\Big(\frac1B\sum_b T_b\Big) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2 .$$

*In words:* adding trees removes the second term, but the correlation between trees sets a floor $\rho\sigma^2$;
bootstrap samples and random feature subsets make the trees less alike (smaller $\rho$).

**Worked example (SIMULATED tumours, 398 train / 171 test, $m = 5$).** Individual randomised trees average 0.925
test accuracy, one ordinary full tree 0.947, the 100-tree majority vote 0.965 (OOB accuracy 0.935). The
correlation between trees rises with $m$: 0.22 ($m = 1$), 0.29 ($m = 5$), 0.39 ($m = 30$, bagging).

**Out-of-bag (OOB) error.** Each bootstrap sample leaves out a fraction $(1 - 1/N)^N \approx 1/e \approx 37\,\%$
of the training cases; predicting each case with the trees that did not see it gives a built-in validation
estimate.

**Knobs.** $m$ (small $m$ = weaker but less correlated trees; $m = M$ is plain bagging; in the lecture's
nested-spheres simulation the forest with $m = 1$, "RF-1", had a slightly lower test error than $m = 3$, "RF-3"
— note that "RF-$k$" means $m = k$, not depth $k$), tree depth (on our tumours depth 3 ≈ full depth, stumps a
little worse), and $B$ (more is never worse, only slower).

**Common confusion.** Adding trees does not overfit (it only shrinks the variance term); unpruned trees are
*wanted* inside a forest (low bias), the averaging takes care of their variance.

> 🎓 **Exam tip.** Know the two sources of randomness (bootstrap samples, random feature subsets at each node), why they lower
    the correlation between trees, and why adding trees does not overfit. Be able to derive
    ρσ² + (1 − ρ)σ²/B in three lines from Var(ΣT_b) = Σ_b Σ_c Cov(T_b, T_c).

---
## 10 · Case study: CHASM — driver or passenger?

> 📍 **Where we are.** Everything is in place: information gain, trees, forests and out-of-bag validation. The
> lecture closes with a case study: how Carter *et al.* (2009) used exactly this toolkit on a real cancer-genomics
> problem.

**The biology.** Cancer progresses by the accumulation of somatic mutations over many cell divisions.
**Driver** mutations are causally implicated in oncogenesis: they give the cell a growth advantage and are positively
selected in the tissue's microenvironment (e.g. mutations that inactivate a tumour suppressor). **Passenger**
mutations have no functional consequence and give no growth advantage. Most mutations in a tumour are passengers.

**The method.** CHASM classifies each mutation as driver or passenger from a table of features of the mutation — of
the kinds a biologist would ask about: how conserved the position is across species, how often such substitutions are
seen in tumours, how radical the amino-acid change is, predicted structural properties, and annotated functional
sites. How informative each feature is about the class can be scored by mutual information — the information gain
of Section 3. The classifier is trained on examples of known driver and passenger mutations. The training set:

- **positives:** 2,488 missense mutations previously identified as functional in oncogenic transformation (breast,
  colorectal and pancreatic tumours);
- **negatives:** synthetic passengers — take the genes mutated at least once in four large sequencing studies
  (colorectal, breast, brain, pancreatic) and generate random missense changes in them.

A **500-tree random forest** is trained on these examples.

**The results.** In cross-validation CHASM's ROC curve lies clearly above those of SIFT and PolyPhen (general
predictors of whether an amino-acid change damages a protein). On held-out experimental
data CHASM called **195 of 196** *TP53* (P53) mutations and **131 of 133** *EGFR* mutations drivers; PolyPhen and SIFT
called 188/196 and 101/133.

🧮 **Refresher — ROC curve and AUC.** A classifier that outputs a score (here, the fraction of trees voting "driver")
can be thresholded anywhere. For each threshold, the **true-positive rate** is the fraction of drivers above it and
the **false-positive rate** the fraction of passengers above it; the **ROC curve** plots one against the other as the
threshold moves. The **AUC** (area under the curve) equals the probability that a randomly chosen driver gets a higher
score than a randomly chosen passenger: 0.5 = coin flip, 1 = perfect ranking.

### 10.1 A SIMULATED mini-CHASM

The course stores no data, so we simulate 900 missense mutations (300 drivers, 600 passengers)
with six interpretable features whose distributions differ between the classes in biologically plausible directions,
plus three pure-noise features. The numbers are invented; only the *mechanics* carry over. (The repository's
[DecisionTreesRandomForests](../../DecisionTreesRandomForests/) notebook trains a forest on **real** TCGA driver and
passenger mutations.)

In [ ]:
# ── 10.1 Simulate driver and passenger missense mutations (SIMULATED) ──
def simulate_mutations(rng, n_driver: int = 300, n_passenger: int = 600):
    """SIMULATED stand-in for CHASM-style features. Returns X, y (1 = driver) and feature names."""
    y = np.r_[np.ones(n_driver, int), np.zeros(n_passenger, int)]
    d = y == 1
    n = len(y)
    conservation = np.where(d, rng.beta(5, 2, n), rng.beta(2.5, 2.5, n))          # 0–1, drivers more conserved
    in_domain = (rng.random(n) < np.where(d, 0.55, 0.25)).astype(int)            # in an annotated functional domain
    recurrence = rng.poisson(np.where(d, 2.0, 0.4))                             # times seen in other tumours
    recurrence += rng.poisson(np.where(d & (in_domain == 1) & (conservation > 0.7), 3.0, 0.0))  # hotspots
    substitution = np.round(np.where(d, rng.normal(-1.0, 2.0, n), rng.normal(0.5, 2.0, n)))   # PAM-like; low = radical
    hydrophobicity_change = np.abs(np.where(d, rng.normal(1.2, 1.0, n), rng.normal(0.8, 1.0, n)))
    buried = np.where(d, rng.beta(3, 2, n), rng.beta(2, 3, n))                  # fraction of residue buried
    noise = rng.normal(0, 1, (n, 3))
    X = np.column_stack([conservation, recurrence, in_domain, substitution, hydrophobicity_change, buried, noise])
    names = ["conservation", "recurrence_in_tumours", "in_functional_domain", "substitution_score",
             "hydrophobicity_change", "buried_fraction", "noise_1", "noise_2", "noise_3"]
    return X, y, names


rng = np.random.default_rng(SEED + 10)
X_mut, y_mut, MUT_FEATS = simulate_mutations(rng)
print("SIMULATED mutations:", X_mut.shape, "| drivers:", y_mut.sum(), "| passengers:", (1 - y_mut).sum())
print(pd.DataFrame(X_mut, columns=MUT_FEATS).groupby(y_mut).mean().rename(index={0: "passenger", 1: "driver"})
      .round(2).T.to_string())

In [ ]:
# ── 10.2 Rank features by mutual information with the class, as in CHASM's feature table ──
def binned_information_gain(x, y, n_bins: int = 4) -> float:
    """Information gain (bits) of a feature after cutting it into quantile bins (discrete features stay as they are)."""
    if len(np.unique(x)) > n_bins:
        x = np.digitize(x, np.quantile(x, np.linspace(0, 1, n_bins + 1)[1:-1]))
    return information_gain(x, y)


mi = pd.Series({f: binned_information_gain(X_mut[:, j], y_mut) for j, f in enumerate(MUT_FEATS)})
print(mi.sort_values(ascending=False).round(4).to_string())

🔍 **What just happened?** Ranking single features by information gain puts recurrence and conservation at the top
and the noise features at the bottom, with values close to (but, because of sampling noise, not exactly) zero — the
same kind of single-feature ranking one would make for CHASM's feature table. (Estimated mutual information can be
slightly positive for a useless feature by chance — a true mutual information is never negative, and it is zero only
when feature and class are independent.) A single-feature ranking ignores interactions, which is what the forest is
for.

In [ ]:
# ── 10.3 A 500-tree forest; out-of-bag ROC curve vs single-feature scores ──
t0 = time.time()
chasm_like = RandomForestClassifier(n_estimators=500, criterion="entropy", max_features="sqrt", oob_score=True,
                                    random_state=SEED, n_jobs=1).fit(X_mut, y_mut)
oob_score = chasm_like.oob_decision_function_[:, 1]          # fraction of 'out-of-bag' trees voting driver
print(f"trained 500 trees in {time.time() - t0:.1f} s; OOB accuracy {chasm_like.oob_score_:.3f}")

scores = {"random forest (OOB)": oob_score, "conservation only": X_mut[:, 0],
          "substitution score only (low = radical)": -X_mut[:, 3]}
fig, ax = plt.subplots(figsize=(4.6, 4.0))
for (name, s), col in zip(scores.items(), ["tab:green", "tab:blue", "tab:orange"]):
    fpr, tpr, _ = roc_curve(y_mut, s)
    ax.plot(fpr, tpr, color=col, label=f"{name}: AUC {roc_auc_score(y_mut, s):.2f}")
ax.plot([0, 1], [0, 1], "k:", lw=0.8)
ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title("SIMULATED mini-CHASM: ROC curves"); ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The forest's out-of-bag ROC curve lies well above those of single features — the analogue
of CHASM's cross-validation result, where CHASM (which combines many features) beats SIFT and PolyPhen (general
predictors of protein damage). The OOB score is an honest held-out estimate: each mutation is scored only by trees that never saw it.
Our AUC values are properties of our invented simulation, not of CHASM.

### 10.4 Which features does the forest use?

In [ ]:
# ── 10.4 Impurity-based vs permutation importance ──
rng = np.random.default_rng(SEED + 104)
tr_m, te_m = train_test_indices(len(y_mut), 0.3, rng, stratify=y_mut)
rf_m = RandomForestClassifier(n_estimators=200, criterion="entropy", max_features="sqrt", random_state=SEED,
                              n_jobs=1).fit(X_mut[tr_m], y_mut[tr_m])
perm = permutation_importance(rf_m, X_mut[te_m], y_mut[te_m], n_repeats=5, random_state=SEED, n_jobs=1)
imp = pd.DataFrame({"impurity (training data)": rf_m.feature_importances_,
                    "permutation (held-out data)": perm.importances_mean}, index=MUT_FEATS)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, col, c in zip(axes, imp.columns, ["tab:purple", "tab:green"]):
    ax.barh(MUT_FEATS[::-1], imp[col].to_numpy()[::-1], color=c)
    ax.set_title(col); ax.axvline(0, color="k", lw=0.6)
axes[0].set_xlabel("mean decrease in entropy (normalised)"); axes[1].set_xlabel("drop in held-out accuracy")
plt.suptitle("SIMULATED mini-CHASM: what the forest relies on", fontsize=10); plt.tight_layout(); plt.show()
print(imp.round(3).to_string())

🔍 **What just happened?** Two common importance measures disagree in an instructive way. **Impurity importance**
adds up the entropy decrease of every split on a feature, on the training data. It gives the three *noise* features
visible importance and ranks the binary *in_functional_domain* low: continuous features offer hundreds of candidate
thresholds, so by chance they can always find a split that helps a little on the training data (the same
many-values bias as the *Day* attribute in Section 3). **Permutation importance** shuffles one feature in held-out
data and measures how much accuracy drops: the noise features fall to about zero. Permutation importance has its own
blind spot: *in_functional_domain* also scores near zero, although it is informative on its own (Section 10.2) —
we simulated extra recurrence for domain hotspots, so once the forest knows the recurrence, shuffling the domain flag
costs little. Correlated or redundant features share (and can hide) each other's importance. Prefer held-out measures,
and remember that importance means "useful for prediction in this data set", not "biologically causal".

✏️ **Your turn.** Delete `recurrence_in_tumours` (column 1) from `X_mut` and refit the forest of 10.3. How much does the
OOB AUC drop, and which feature takes over? Why might a feature like recurrence be *circular* in a real study (hint:
how were the known drivers discovered)?

### 📘 Lecture notes — case study: CHASM, driver vs passenger mutations

**Biology.** Cancer progresses by accumulating somatic mutations. **Driver** mutations are causally implicated in
oncogenesis: they give cancer cells a growth advantage and are positively selected in the microenvironment of the
tissue (e.g. mutations that de-activate tumour-suppressor genes). **Passenger** mutations are somatic mutations
with no functional consequence and no growth advantage.

**Method (Carter *et al.* 2009).** Classify each mutation as driver or passenger from a table of features of the
mutation (conservation, frequency in tumours, severity of the amino-acid change, structure, functional sites).
Train a **random forest of 500 trees** on examples of known driver and passenger mutations.

**Training set.** Positives: 2,488 missense mutations previously identified as playing a functional role in
oncogenic transformation (breast, colorectal, pancreatic tumours). Negatives: (1) select genes mutated at least
once in four large sequencing studies (colorectal, breast, brain, pancreatic); (2) generate **synthetic**
passenger missense mutations in them.

**Results.** Cross-validated ROC curves: CHASM clearly above SIFT and PolyPhen. Held-out experimental
data: CHASM called 195/196 *TP53* and 131/133 *EGFR* mutations drivers; PolyPhen and SIFT 188/196 and 101/133.

**Our simulation.** A mini-CHASM on 900 SIMULATED mutations: recurrence and conservation lead the
information-gain ranking, the 500-tree forest's out-of-bag ROC curve beats single features, and impurity
importance flatters continuous noise features where permutation importance does not.

**Common confusions.** AUC = probability that a random driver scores above a random passenger (0.5 = chance);
synthetic negatives may differ from real passengers in ways a forest can exploit; feature importance means
"useful for prediction", not "causal".

> 🎓 **Exam tip.** Be able to explain how the negative training examples were constructed and why that matters (synthetic
    passengers may differ from real passengers in ways the forest can exploit), what an AUC of, say, 0.9 means, and which
    two kinds of randomness the 500-tree forest uses.

---
## 📌 Key takeaways

- A **decision tree** is a flowchart: internal nodes test attributes, branches are values or threshold outcomes,
  leaves predict a class (or class frequencies). It is a **non-probabilistic** learner, interpretable, able to
  express interactions, and cheap to evaluate.
- **Entropy** $H = -\sum p\log_2 p$ measures label impurity in bits; **information gain** $H(Y) - H(Y\mid A)$ — the
  mutual information between attribute and label — picks the split. On PlayTennis: 0.940 bits at the root, gains
  0.247 (Outlook), 0.152 (Humidity), 0.048 (Wind), 0.029 (Temperature).
- **ID3** grows the tree greedily, top-down and recursively until nodes are pure; each tree is a set of
  **IF-THEN rules**, one per leaf.
- Pure-leaf trees **overfit** noisy labels. Remedies: early stopping, or growing the full tree and **pruning** it
  (reduced-error pruning on a *validation* set; report accuracy on an untouched test set).
- Continuous attributes are split at **thresholds** (midpoints at label changes); trees carve **axis-parallel
  rectangles** and need staircases for oblique boundaries.
- Trees are **high-variance**: small changes in the data restructure them. Averaging $B$ trees with correlation $\rho$
  leaves variance $\rho\sigma^2 + (1-\rho)\sigma^2/B$.
- A **random forest** = bootstrap samples + $m \ll M$ random candidate features per split + unpruned trees + majority
  vote; the out-of-bag error is a built-in validation estimate. $m$, depth and the number of trees are tuning choices.
- **CHASM** applied a 500-tree forest to distinguish driver from passenger mutations and outperformed SIFT and
  PolyPhen — with the caveats that come with synthetic negatives and importance ≠ causation.

---
## ✅ Checkpoint

1. A node contains 6 *Yes* and 2 *No* examples. What is its entropy?
   <details><summary>Answer</summary>$p_\oplus = 0.75$, $p_\ominus = 0.25$: $H = -0.75\log_2 0.75 - 0.25\log_2 0.25 = 0.311 + 0.5 = 0.811$ bits (the same as the Wind = Weak branch of PlayTennis, [6+, 2−]).</details>
2. Wind splits PlayTennis into Weak [6+, 2−] and Strong [3+, 3−]. Compute Gain(S, Wind).
   <details><summary>Answer</summary>$H(Y\mid\text{Wind}) = \tfrac{8}{14}\cdot 0.811 + \tfrac{6}{14}\cdot 1.000 = 0.892$; gain $= 0.940 - 0.892 = 0.048$ bits.</details>
3. Why does an ID-like attribute (a unique sample barcode) get the largest possible information gain, and what is wrong with splitting on it?
   <details><summary>Answer</summary>Every value contains one example, so every child is pure and H(Y|A) = 0: the gain equals H(Y). But the split memorises the training set and says nothing about a new sample (whose barcode was never seen). C4.5's gain ratio divides by H(A) to penalise such attributes; better still, never use identifiers as features.</details>
4. In reduced-error pruning, why can't the accuracy on the pruning set be reported as the final performance?
   <details><summary>Answer</summary>The pruning set was used to choose which subtrees to remove, so the final tree was selected to do well on it; its accuracy there is optimistically biased. An untouched test set (or nested cross-validation) is needed.</details>
5. A sorted continuous attribute has values 1.2, 1.9, 2.4, 3.0, 3.8 with labels −, −, +, +, −. Which thresholds are candidates?
   <details><summary>Answer</summary>Midpoints at label changes: (1.9 + 2.4)/2 = 2.15 and (3.0 + 3.8)/2 = 3.4. By the Fayyad–Irani result the best information-gain threshold is one of these.</details>
6. Trees in a forest have variance $\sigma^2 = 0.04$ and pairwise correlation $\rho = 0.25$. What is the variance of the average of 100 trees, and the limit as $B\to\infty$?
   <details><summary>Answer</summary>$\rho\sigma^2 + (1-\rho)\sigma^2/B = 0.01 + 0.75\cdot 0.04/100 = 0.0103$; the limit is $\rho\sigma^2 = 0.01$. Lowering $\rho$ (smaller $m$) lowers the floor.</details>
7. Read the code: in `random_forest`, which line creates the bootstrap sample and which argument creates the random feature subsets?
   <details><summary>Answer</summary>`boot = rng.integers(0, len(y), len(y))` draws N indices with replacement (the bootstrap). The argument `m_features=m` passed to `grow_tree` makes each node call `rng.choice(X.shape[1], m_features, replace=False)` — a fresh random subset of m features at every split.</details>
8. Why do random forests not need pruning, while single trees do?
   <details><summary>Answer</summary>A single unpruned tree has low bias but high variance. Averaging many decorrelated trees reduces the variance term (ρσ² + (1 − ρ)σ²/B) while keeping the low bias, so fully grown trees are what you want inside a forest.</details>

## 🧠 Questions & discussion

1. Information gain, the Gini index (used by CART and scikit-learn's default) and misclassification rate are all impurity measures. Plot all three for a binary node as a function of $p_\oplus$. Why is misclassification rate a poor splitting criterion even though it is what we ultimately care about?
2. In a GWAS with 500,000 SNPs and 2,000 individuals, would you expect a single decision tree to find an epistatic pair of SNPs whose marginal effects are both zero? What does greedy splitting imply here?
3. CHASM's negatives are synthetic passengers drawn in genes that are mutated in tumours. Name one way in which synthetic passengers might differ systematically from real passengers, and how that could inflate the reported accuracy.
4. A clinician wants a model she can audit. Would you give her a pruned tree, a random forest, or a logistic regression? What would you lose with each choice?
5. Trees split on one feature at a time. How would you modify the algorithm to allow oblique splits ($w^\top x \le \theta$)? What would you gain and lose?

## 🚀 Bonus — going further

- Real data: the repository's [DecisionTreesRandomForests](../../DecisionTreesRandomForests/) notebook trains a from-scratch CART forest (Gini impurity) on real TCGA breast-cancer driver/passenger mutations, compares impurity and permutation importance, and tests generalisation to melanoma.
- Replace entropy by the Gini index $1-\sum_c p_c^2$ in `best_threshold` and compare the trees on the simulated tumour data. How often do the two criteria choose a different root split?
- Implement **gradient boosting** with depth-1 trees (stumps): fit each new stump to the residual errors of the current ensemble. Compare with the depth-1 forest of Section 9.4b (ESL Chapter 10).
- Use the forest's OOB estimate to choose $m$ and the minimum leaf size on the SIMULATED leukaemia data (`simulate_leukemia_expression` in the course toolkit; 72 patients × 3,571 genes, ALL vs AML): which genes appear most often near the roots, and are they among the true marker genes (`true_effect`)?
- Read Breiman (2001), Section 2: prove the bound PE* ≤ ρ̄(1 − s²)/s² relating forest error to tree strength s and correlation ρ̄, and estimate both for our forests.
- Level 2: random forests on molecular fingerprints are the classic baseline of [Module 40](40_molecular_ml_drug_discovery.ipynb), and [Module 42](42_trustworthy_ml.ipynb) shows how leakage (e.g. a feature that encodes how the labels were made, like recurrence in CHASM) inflates accuracy.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 10 · Decision trees, random forests and cancer driver mutations
Input: PlayTennis (14 + D15 days); SIMULATED genotype study (600 × 10 SNPs); SIMULATED flow-cytometry
       cells (150 × FSC, SSC); SIMULATED breast tumours (569 × 30); SIMULATED driver/passenger mutations (900 × 9)
Output: ID3 tree and rules; pruned tree; numeric trees; random forests; OOB ROC curves
Method: entropy-based greedy tree growing (ID3/C4.5-style), reduced-error pruning, bagging + random
        feature subsets (random forest), out-of-bag validation
Key equation: Gain(S, A) = H_S(Y) − Σ_v |S_v|/|S| · H_{S_v}(Y);  Var(avg) = ρσ² + (1 − ρ)σ²/B
Key parameters: max depth / pruning set; number of trees B; features per split m; tree depth in a forest
What changed when I modified the label-noise rate / m / tree depth: …
One failure mode: identifier-like or many-valued attributes win information gain; trees overfit noisy labels
Connection to the 02-620 topics (which idea of the lecture this illustrates):
              greedy top-down tree learning with information gain; overfitting and reduced-error pruning;
              thresholds and rectangles; high variance; random forests; CHASM
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the material of its lecture on decision trees and random forests.
- Mitchell T. M. (1997) *Machine Learning*, Chapter 3 (Decision tree learning). McGraw-Hill.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*, Sections 16.1–16.2. MIT Press.
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed., Sections 9.2 (trees), 8.7 (bagging) and Chapter 15 (random forests). Springer.
- Quinlan J. R. (1986) Induction of decision trees. *Machine Learning* 1:81–106.
- Quinlan J. R. (1993) *C4.5: Programs for Machine Learning*. Morgan Kaufmann.
- Fayyad U. M., Irani K. B. (1992) On the handling of continuous-valued attributes in decision tree generation. *Machine Learning* 8:87–102.
- Breiman L. (2001) Random forests. *Machine Learning* 45:5–32.
- Carter H., Chen S., Isik L., Tyekucheva S., Velculescu V. E., Kinzler K. W., Vogelstein B., Karchin R. (2009) Cancer-specific high-throughput annotation of somatic mutations: computational prediction of driver missense mutations. *Cancer Research* 69:6660–6667.
- Wolberg W. H., Street W. N., Mangasarian O. L. (1995) Wisconsin Diagnostic Breast Cancer data set, UCI Machine Learning Repository (the study `simulate_tumor_features` imitates).
- Fisher R. A. (1936) The use of multiple measurements in taxonomic problems. *Annals of Eugenics* 7:179–188 (the iris flowers of the lecture's rectangle example; `simulate_blood_cells` plays their role here).
- Shapiro H. M. (2003) *Practical Flow Cytometry*, 4th ed. Wiley-Liss — forward and side scatter, gating.

## ➡️ Next

Trees partition space with many small axis-parallel cuts. [Module 11](11_support_vector_machines.ipynb)
returns to a single boundary — but chooses it by a new principle, the **maximum margin**, and then bends it
with kernels: support vector machines.